## Liar2 Dataset_loading class

In [5]:
"""
liar_preprocessor.py

One class, original LIAR dataset only (Wang, 2017) -- structured the same
way as Liar2Preprocessor (every step its own method, official_split
preserved, opt-in dedup/conflict auditing), with ONE deliberate structural
difference: LIAR has no date column, so nothing here does date parsing,
chronological sorting, time_rank, domain buckets, or rolling windows.
Only a 70/15/15 RANDOM STRATIFIED re-split is offered alongside the
official file-based split -- there is no "chronological_split" method on
this class, and there cannot be one without a date field to sort by.

LIAR's raw TSV format (no header row) has 14 columns, in this order:
    id, label, statement, subject, speaker, job_title, state_info,
    party_affiliation, barely_true_count, false_count, half_true_count,
    mostly_true_count, pants_on_fire_count, context
Confirmed against the actual uploaded train.tsv/valid.tsv/test.tsv.

Note LIAR's schema differs from LIAR2's in a few ways relevant to
preprocessing:
    - No `date` column at all (LIAR2 has one) -> no time-based anything.
    - No `justification` column (LIAR2 has one) -> nothing to exclude there.
    - No `speaker_description` column (LIAR2 has one).
    - Has `job_title` and `party_affiliation` columns that LIAR2 does not.
    - Credibility-history counts use different column names but the same
      underlying concept (barely_true_count, false_count, half_true_count,
      mostly_true_count, pants_on_fire_count) -- these are excluded from
      model text and dropped entirely, for the same leakage reason as in
      Liar2Preprocessor.

Usage
-----
    from liar_preprocessor import LiarPreprocessor

    p = LiarPreprocessor()

    df = p.load_raw(train_path, valid_path, test_path)
    df = p.clean_statement_text(df)
    df = p.normalize_labels(df)
    df = p.filter_invalid_rows(df)
    df = p.build_text_modes(df)
    df = p.drop_leakage_columns(df)

    # or all at once:
    df = p.run_pipeline(train_path, valid_path, test_path)

    # only re-split available (no chronological_split -- no dates exist):
    train_idx, valid_idx, test_idx = p.random_stratified_split(df, seed=42)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


class LiarPreprocessor:

    RAW_COLUMNS = [
        "id", "label", "statement", "subject", "speaker", "job_title",
        "state_info", "party_affiliation", "barely_true_count", "false_count",
        "half_true_count", "mostly_true_count", "pants_on_fire_count", "context",
    ]

    LABEL_ORDER = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
    NUMERIC_LABEL_MAP = {0: "pants-fire", 1: "false", 2: "barely-true",
                         3: "half-true", 4: "mostly-true", 5: "true"}
    # Included in model text (restricted metadata, not leakage-risk):
    SAFE_METADATA_COLUMNS = ["subject", "speaker", "job_title", "state_info",
                             "party_affiliation", "context"]
    # Dropped entirely -- speaker credibility-history counts are leakage risks:
    EXCLUDED_COLUMNS = ["barely_true_count", "false_count", "half_true_count",
                        "mostly_true_count", "pants_on_fire_count"]

    def __init__(
        self,
        train_ratio: float = 0.70,
        valid_ratio: float = 0.15,
        test_ratio: float = 0.15,
        seed: int = 42,
    ):
        assert abs((train_ratio + valid_ratio + test_ratio) - 1.0) < 1e-9, \
            "train_ratio + valid_ratio + test_ratio must sum to 1.0"
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio
        self.seed = seed
        self.label_to_id = {label: i for i, label in enumerate(self.LABEL_ORDER)}

    # ================================================================
    # Step 1: Load, tagging each row with its OFFICIAL file-based split
    # ================================================================

    def load_raw(self, train_path, valid_path, test_path) -> pd.DataFrame:
        """Load the three official LIAR TSVs (no header row in the raw
        files) and concatenate them, tagging each row with
        `official_split` = "train"/"valid"/"test" based on which file it
        came from. `row_order` records original concatenation order --
        used later as the tie-break for duplicate removal, since LIAR has
        no date column to sort by.
        """
        frames = []
        for path, split_name in [(train_path, "train"), (valid_path, "valid"), (test_path, "test")]:
            part = pd.read_csv(path, sep="\t", header=None, names=self.RAW_COLUMNS, dtype=str)
            part["official_split"] = split_name
            part["source_file"] = Path(path).name
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)
        df["row_order"] = np.arange(len(df))

        print("Loaded raw LIAR rows by official_split:")
        print(df["official_split"].value_counts().reindex(["train", "valid", "test"]))
        return df

    # ================================================================
    # Step 2: Text cleaning
    # ================================================================

    @staticmethod
    def clean_text(x) -> str:
        if pd.isna(x):
            return ""
        return re.sub(r"\s+", " ", str(x)).strip()

    def clean_statement_text(self, df: pd.DataFrame) -> pd.DataFrame:
        """Whitespace-collapse and strip the `statement` column."""
        df = df.copy()
        df["statement"] = df["statement"].apply(self.clean_text)
        return df

    # ================================================================
    # Step 3: Label normalization
    # ================================================================

    def _normalize_single_label(self, x):
        if pd.isna(x):
            return np.nan
        try:
            if str(x).strip().replace(".0", "").isdigit():
                xi = int(float(x))
                if xi in self.NUMERIC_LABEL_MAP:
                    return self.NUMERIC_LABEL_MAP[xi]
        except Exception:
            pass
        x = str(x).strip().lower().replace("_", "-")
        x = re.sub(r"\s+", "-", x)
        mapping = {
            "pants-fire": "pants-fire", "pants-on-fire": "pants-fire", "pantsonfire": "pants-fire",
            "false": "false",
            "barely-true": "barely-true", "barelytrue": "barely-true",
            "half-true": "half-true", "halftrue": "half-true",
            "mostly-true": "mostly-true", "mostlytrue": "mostly-true",
            "true": "true",
        }
        return mapping.get(x, x)

    def normalize_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """Map raw label (LIAR's labels are already text, but numeric 0-5
        is also handled for consistency with LiarPreprocessor's sibling
        class) to the canonical 6-class label string. Invalid/unmapped
        labels are left as-is here -- use filter_invalid_rows() to drop
        them."""
        df = df.copy()
        df["label"] = df["label"].apply(self._normalize_single_label)
        return df

    # ================================================================
    # Step 4: Drop empty statements / unmapped labels, assign label_id
    # ================================================================

    def filter_invalid_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop rows with an empty statement or a label outside
        LABEL_ORDER, then assign the integer `label_id`. Prints how many
        rows were removed at each stage."""
        n_start = len(df)

        df = df[df["statement"].str.len() > 0].copy()
        n_after_empty = len(df)

        df = df[df["label"].isin(self.LABEL_ORDER)].copy()
        n_after_label = len(df)

        df["label_id"] = df["label"].map(self.label_to_id).astype(int)

        print(f"filter_invalid_rows: {n_start} -> {n_after_empty} "
              f"(dropped {n_start - n_after_empty} empty statements) -> "
              f"{n_after_label} (dropped {n_after_empty - n_after_label} unmappable labels)")
        return df.reset_index(drop=True)

    # ================================================================
    # Step 5: Build both text-input modes
    # ================================================================

    def _build_claim_only(self, row) -> str:
        return f"claim: {row['statement']}"

    def _build_claim_plus_safe_metadata(self, row) -> str:
        parts = [f"claim: {row['statement']}"]
        for col in self.SAFE_METADATA_COLUMNS:
            if col in row.index:
                val = self.clean_text(row.get(col, ""))
                if val:
                    parts.append(f"{col}: {val}")
        return " [SEP] ".join(parts)

    def build_text_modes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adds two columns:
            model_text__claim_only
            model_text__claim_plus_safe_metadata
        so both conditions of the text-input ablation are available at
        once."""
        df = df.copy()
        df["model_text__claim_only"] = df.apply(self._build_claim_only, axis=1)
        df["model_text__claim_plus_safe_metadata"] = df.apply(self._build_claim_plus_safe_metadata, axis=1)
        return df

    # ================================================================
    # Step 6: Drop leakage-risk columns entirely
    # ================================================================

    def drop_leakage_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drops the speaker credibility-history count columns entirely
        (not just excludes them from model text), so they can't be
        accidentally used downstream. (LIAR has no `justification` column
        to drop -- that's LIAR2-specific.)"""
        present = [c for c in self.EXCLUDED_COLUMNS if c in df.columns]
        df = df.drop(columns=present)
        print(f"drop_leakage_columns: removed {present if present else '(none present)'}")
        return df

    # ================================================================
    # Duplicate / conflicting-label handling (opt-in, NOT called by
    # run_pipeline automatically)
    # ================================================================

    @staticmethod
    def _normalize_for_overlap(x: str) -> str:
        """Aggressive normalization used ONLY for duplicate detection,
        never for the text fed to a model."""
        x = str(x).lower()
        x = re.sub(r"[^a-z0-9\s]", "", x)
        return re.sub(r"\s+", " ", x).strip()

    def audit_duplicate_statements(self, df: pd.DataFrame) -> pd.DataFrame:
        """Diagnostic only -- does NOT modify df. Groups rows by normalized
        statement text and reports, per group: row count, distinct labels
        present, and whether the group has a genuine label conflict."""
        work = df.copy()
        work["_norm"] = work["statement"].apply(self._normalize_for_overlap)

        grouped = work.groupby("_norm").agg(
            n_rows=("label", "size"),
            distinct_labels=("label", lambda s: sorted(set(s))),
        ).reset_index()
        grouped["is_duplicate_group"] = grouped["n_rows"] > 1
        grouped["has_label_conflict"] = grouped["distinct_labels"].apply(len) > 1
        return grouped

    def identify_conflicting_label_groups(self, audit_df: pd.DataFrame) -> pd.DataFrame:
        """Subset of audit_duplicate_statements() output where the same
        normalized claim carries more than one distinct label."""
        return audit_df[audit_df["has_label_conflict"]].copy()

    def drop_duplicate_statements(
        self,
        df: pd.DataFrame,
        policy: str = "keep_first",
        audit_df=None,
    ) -> pd.DataFrame:
        """
        NOT called automatically by run_pipeline -- opt-in only, since it
        changes row count. Since LIAR has no date column, "earliest" here
        means earliest by `row_order` (original file concatenation order:
        train rows before valid rows before test rows) rather than by
        timestamp.

        policy:
            "keep_first"              -> for every duplicate group (exact
                                          or conflicting-label), keep only
                                          the row with the lowest row_order.
            "drop_conflicting_groups" -> conflicting-label groups are
                                          DROPPED ENTIRELY; non-conflicting
                                          duplicate groups still keep the
                                          row_order-earliest row.
        """
        if audit_df is None:
            audit_df = self.audit_duplicate_statements(df)

        work = df.copy()
        work["_norm"] = work["statement"].apply(self._normalize_for_overlap)

        conflicts = self.identify_conflicting_label_groups(audit_df)
        n_conflict_groups = len(conflicts)
        n_conflict_rows = int(work["_norm"].isin(conflicts["_norm"]).sum())
        if n_conflict_groups > 0:
            print(f"drop_duplicate_statements: {n_conflict_groups} conflicting-label group(s) "
                  f"covering {n_conflict_rows} row(s) found.")

        if policy == "drop_conflicting_groups" and n_conflict_groups > 0:
            work = work[~work["_norm"].isin(conflicts["_norm"])].copy()

        before = len(work)
        sort_col = "row_order" if "row_order" in work.columns else work.index
        work = work.sort_values(sort_col).drop_duplicates(subset=["_norm"], keep="first")
        after = len(work)
        print(f"drop_duplicate_statements ({policy}): {before} -> {after} (removed {before - after})")

        return work.drop(columns=["_norm"]).reset_index(drop=True)

    # ================================================================
    # Re-split (random stratified ONLY -- no chronological_split exists,
    # because LIAR has no date column to sort by)
    # ================================================================

    def random_stratified_split(self, df: pd.DataFrame, seed: int = None):
        """70/15/15 stratified-by-label split. This is the ONLY re-split
        offered on this class -- unlike Liar2Preprocessor, there is no
        chronological_split() here, because LIAR has no date field. If you
        need a temporal analysis, it has to be done on LIAR2, not LIAR."""
        seed = self.seed if seed is None else seed
        all_idx = np.arange(len(df))
        y = df["label_id"].values

        train_idx, temp_idx = train_test_split(
            all_idx, train_size=self.train_ratio, random_state=seed, stratify=y,
        )
        y_temp = y[temp_idx]
        valid_relative_ratio = self.valid_ratio / (self.valid_ratio + self.test_ratio)
        valid_idx, test_idx = train_test_split(
            temp_idx, train_size=valid_relative_ratio, random_state=seed, stratify=y_temp,
        )
        return np.sort(train_idx), np.sort(valid_idx), np.sort(test_idx)

    # ================================================================
    # Full pipeline
    # ================================================================

    def run_pipeline(self, train_path, valid_path, test_path) -> pd.DataFrame:
        """Runs every step in sequence:
            load_raw -> clean_statement_text -> normalize_labels ->
            filter_invalid_rows -> build_text_modes -> drop_leakage_columns
        No date parsing, no time_rank, no chronological_split step exists
        for this dataset -- see class docstring.
        """
        df = self.load_raw(train_path, valid_path, test_path)
        df = self.clean_statement_text(df)
        df = self.normalize_labels(df)
        df = self.filter_invalid_rows(df)
        df = self.build_text_modes(df)
        df = self.drop_leakage_columns(df)

        print(f"\nrun_pipeline complete: {len(df)} final rows.")
        print("Official split counts (post-cleaning):")
        print(df["official_split"].value_counts().reindex(["train", "valid", "test"]))
        return df


if __name__ == "__main__":
    print("liar_preprocessor.py loaded. Class: LiarPreprocessor "
          "(no chronological_split -- LIAR has no date column)")

liar_preprocessor.py loaded. Class: LiarPreprocessor (no chronological_split -- LIAR has no date column)


In [6]:
"""
liar2_preprocessor.py

One class, LIAR2 only. Every preprocessing step is its own method so you
can call them individually and inspect intermediate output, instead of
only seeing the final combined result.

Two distinct splits are tracked, on purpose, because they answer different
questions:
    - `official_split` : which of train.csv/valid.csv/test.csv a row came
                          from. This is LIAR2's own distributed split
                          (~80/10/10). Useful for reporting results against
                          the split other papers on LIAR2 use.
    - chronological_split() / random_stratified_split() : the project's
      OWN 70/15/15 re-split of the full recombined, chronologically-sorted
      dataset, matching liar2-split-and-input-ablation-experiments*.ipynb
      and da-minilm-main-model-v1-0.ipynb. This is the split your reported
      DA-MiniLM-RWAC / MiniLM-LogReg-RW numbers were actually evaluated on.

These are NOT the same partition (verified: the official split is
18369/2297/2296 = ~80/10/10; the canonical re-split ratios are 70/15/15
over the recombined data). Don't mix them up when reporting results.

Usage
-----
    from liar2_preprocessor import Liar2Preprocessor

    p = Liar2Preprocessor()

    # step by step, so you can inspect each stage:
    df = p.load_raw(train_path, valid_path, test_path)
    df = p.clean_statement_text(df)
    df = p.normalize_labels(df)
    df = p.filter_invalid_rows(df)
    df = p.parse_and_sort_by_date(df)
    df = p.build_text_modes(df)
    df = p.drop_leakage_columns(df)
    df = p.add_time_rank(df)

    # or all at once:
    df = p.run_pipeline(train_path, valid_path, test_path)

    # re-splits, computed on the fully-cleaned df:
    train_idx, valid_idx, test_idx = p.chronological_split(df)
    train_idx, valid_idx, test_idx = p.random_stratified_split(df, seed=42)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


class Liar2Preprocessor:

    LABEL_ORDER = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
    NUMERIC_LABEL_MAP = {0: "pants-fire", 1: "false", 2: "barely-true",
                         3: "half-true", 4: "mostly-true", 5: "true"}
    # Included in model text (restricted metadata, not leakage-risk):
    SAFE_METADATA_COLUMNS = ["subject", "speaker", "speaker_description", "state_info", "context"]
    # Dropped entirely -- justification and speaker credibility-history counts are leakage risks:
    EXCLUDED_COLUMNS = ["justification", "true_counts", "mostly_true_counts", "half_true_counts",
                        "mostly_false_counts", "false_counts", "pants_on_fire_counts"]

    def __init__(
        self,
        train_ratio: float = 0.70,
        valid_ratio: float = 0.15,
        test_ratio: float = 0.15,
        date_coverage_threshold: float = 0.80,
        seed: int = 42,
    ):
        assert abs((train_ratio + valid_ratio + test_ratio) - 1.0) < 1e-9, \
            "train_ratio + valid_ratio + test_ratio must sum to 1.0"
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio
        self.date_coverage_threshold = date_coverage_threshold
        self.seed = seed
        self.label_to_id = {label: i for i, label in enumerate(self.LABEL_ORDER)}

    # ================================================================
    # Step 1: Load, tagging each row with its OFFICIAL file-based split
    # ================================================================

    def load_raw(self, train_path, valid_path, test_path) -> pd.DataFrame:
        """Load the three official LIAR2 CSVs and concatenate them, tagging
        each row with `official_split` = "train"/"valid"/"test" based on
        which file it came from. Nothing else is cleaned yet -- this is
        the raw concatenation, useful for auditing raw label distributions
        or row counts per official split before any cleaning happens.
        """
        frames = []
        for path, split_name in [(train_path, "train"), (valid_path, "valid"), (test_path, "test")]:
            part = pd.read_csv(path)
            part["official_split"] = split_name
            part["source_file"] = Path(path).name
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)

        print("Loaded raw LIAR2 rows by official_split:")
        print(df["official_split"].value_counts().reindex(["train", "valid", "test"]))
        return df

    # ================================================================
    # Step 2: Text cleaning
    # ================================================================

    @staticmethod
    def clean_text(x) -> str:
        if pd.isna(x):
            return ""
        return re.sub(r"\s+", " ", str(x)).strip()

    def clean_statement_text(self, df: pd.DataFrame) -> pd.DataFrame:
        """Whitespace-collapse and strip the `statement` column."""
        df = df.copy()
        df["statement"] = df["statement"].apply(self.clean_text)
        return df

    # ================================================================
    # Step 3: Label normalization
    # ================================================================

    def _normalize_single_label(self, x):
        if pd.isna(x):
            return np.nan
        try:
            if str(x).strip().replace(".0", "").isdigit():
                xi = int(float(x))
                if xi in self.NUMERIC_LABEL_MAP:
                    return self.NUMERIC_LABEL_MAP[xi]
        except Exception:
            pass
        x = str(x).strip().lower().replace("_", "-")
        x = re.sub(r"\s+", "-", x)
        mapping = {
            "pants-fire": "pants-fire", "pants-on-fire": "pants-fire", "pantsonfire": "pants-fire",
            "false": "false",
            "barely-true": "barely-true", "barelytrue": "barely-true",
            "half-true": "half-true", "halftrue": "half-true",
            "mostly-true": "mostly-true", "mostlytrue": "mostly-true",
            "true": "true",
        }
        return mapping.get(x, x)

    def normalize_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """Map raw label (numeric 0-5 or text, with spelling variants) to
        the canonical 6-class label string. Invalid/unmapped labels are
        left as-is here -- use filter_invalid_rows() to drop them."""
        df = df.copy()
        df["label"] = df["label"].apply(self._normalize_single_label)
        return df

    # ================================================================
    # Step 4: Drop empty statements / unmapped labels, assign label_id
    # ================================================================

    def filter_invalid_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop rows with an empty statement or a label outside LABEL_ORDER,
        then assign the integer `label_id`. Prints how many rows were
        removed at each stage so you can see exactly what got dropped."""
        n_start = len(df)

        df = df[df["statement"].str.len() > 0].copy()
        n_after_empty = len(df)

        df = df[df["label"].isin(self.LABEL_ORDER)].copy()
        n_after_label = len(df)

        df["label_id"] = df["label"].map(self.label_to_id).astype(int)

        print(f"filter_invalid_rows: {n_start} -> {n_after_empty} "
              f"(dropped {n_start - n_after_empty} empty statements) -> "
              f"{n_after_label} (dropped {n_after_empty - n_after_label} unmappable labels)")
        return df.reset_index(drop=True)

    # ================================================================
    # Step 5: Date parsing + chronological sort (with coverage fallback)
    # ================================================================

    def parse_and_sort_by_date(self, df: pd.DataFrame) -> pd.DataFrame:
        """Parse `date` into `date_parsed`. If more than
        `date_coverage_threshold` fraction of rows have a valid date,
        sort the whole frame chronologically; otherwise fall back to the
        original row order (matches the canonical notebooks' behavior).
        Assigns `claim_id` as the post-sort row index either way.
        Sets `df.attrs["sort_source"]` and `df.attrs["date_coverage"]`.
        """
        df = df.copy()
        df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce") if "date" in df.columns else pd.NaT

        n_total = len(df)
        n_dated = int(df["date_parsed"].notna().sum())
        coverage = n_dated / n_total if n_total else 0.0

        if coverage > self.date_coverage_threshold:
            df = df.sort_values("date_parsed").reset_index(drop=True)
            sort_source = "date"
        else:
            df = df.reset_index(drop=True)
            sort_source = "row_order"

        df["claim_id"] = np.arange(len(df))
        df.attrs["sort_source"] = sort_source
        df.attrs["date_coverage"] = coverage

        print(f"parse_and_sort_by_date: {n_dated}/{n_total} rows have a valid date "
              f"({coverage:.2%}) -> sort_source = '{sort_source}'")
        return df

    # ================================================================
    # Step 6: Build both text-input modes (for the claim-only vs
    # claim+metadata ablation)
    # ================================================================

    def _build_claim_only(self, row) -> str:
        return f"claim: {row['statement']}"

    def _build_claim_plus_safe_metadata(self, row) -> str:
        parts = [f"claim: {row['statement']}"]
        for col in self.SAFE_METADATA_COLUMNS:
            if col in row.index:
                val = self.clean_text(row.get(col, ""))
                if val:
                    parts.append(f"{col}: {val}")
        return " [SEP] ".join(parts)

    def build_text_modes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adds two columns:
            model_text__claim_only
            model_text__claim_plus_safe_metadata
        so both conditions of the text-input ablation are available at
        once without re-running preprocessing."""
        df = df.copy()
        df["model_text__claim_only"] = df.apply(self._build_claim_only, axis=1)
        df["model_text__claim_plus_safe_metadata"] = df.apply(self._build_claim_plus_safe_metadata, axis=1)
        return df

    # ================================================================
    # Step 7: Drop leakage-risk columns entirely
    # ================================================================

    def drop_leakage_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drops `justification` and all speaker credibility-history count
        columns entirely (not just excludes them from model text), so they
        can't be accidentally used downstream."""
        present = [c for c in self.EXCLUDED_COLUMNS if c in df.columns]
        df = df.drop(columns=present)
        print(f"drop_leakage_columns: removed {present if present else '(none present)'}")
        return df

    # ================================================================
    # Duplicate / conflicting-label handling (opt-in, NOT called by
    # run_pipeline automatically -- see note below)
    # ================================================================

    @staticmethod
    def _normalize_for_overlap(x: str) -> str:
        """Aggressive normalization used ONLY for duplicate detection,
        never for the text fed to a model."""
        x = str(x).lower()
        x = re.sub(r"[^a-z0-9\s]", "", x)
        return re.sub(r"\s+", " ", x).strip()

    def audit_duplicate_statements(self, df: pd.DataFrame) -> pd.DataFrame:
        """Diagnostic only -- does NOT modify df. Groups rows by normalized
        statement text and reports, per group: row count, distinct labels
        present, and whether the group has a genuine label conflict (same
        claim, different verdicts)."""
        work = df.copy()
        work["_norm"] = work["statement"].apply(self._normalize_for_overlap)

        grouped = work.groupby("_norm").agg(
            n_rows=("label", "size"),
            distinct_labels=("label", lambda s: sorted(set(s))),
        ).reset_index()
        grouped["is_duplicate_group"] = grouped["n_rows"] > 1
        grouped["has_label_conflict"] = grouped["distinct_labels"].apply(len) > 1
        return grouped

    def identify_conflicting_label_groups(self, audit_df: pd.DataFrame) -> pd.DataFrame:
        """Subset of audit_duplicate_statements() output where the same
        normalized claim carries more than one distinct label."""
        return audit_df[audit_df["has_label_conflict"]].copy()

    def drop_duplicate_statements(
        self,
        df: pd.DataFrame,
        policy: str = "keep_first",
        audit_df=None,
    ) -> pd.DataFrame:
        """
        NOT part of the canonical pipeline -- the source notebooks this
        project is based on (liar2-split-and-input-ablation-experiments*
        .ipynb, da-minilm-main-model-v1-0.ipynb) contain no deduplication
        step, so calling this will change your row count relative to any
        previously reported results. Call it deliberately, not by default.

        policy:
            "keep_first"              -> for every duplicate group (exact
                                          or conflicting-label), keep only
                                          the chronologically earliest row.
            "drop_conflicting_groups" -> conflicting-label groups are
                                          DROPPED ENTIRELY (all rows in the
                                          group removed); non-conflicting
                                          duplicate groups still keep the
                                          earliest row.
        """
        if audit_df is None:
            audit_df = self.audit_duplicate_statements(df)

        work = df.copy()
        work["_norm"] = work["statement"].apply(self._normalize_for_overlap)

        conflicts = self.identify_conflicting_label_groups(audit_df)
        n_conflict_groups = len(conflicts)
        n_conflict_rows = int(work["_norm"].isin(conflicts["_norm"]).sum())
        if n_conflict_groups > 0:
            print(f"drop_duplicate_statements: {n_conflict_groups} conflicting-label group(s) "
                  f"covering {n_conflict_rows} row(s) found.")

        if policy == "drop_conflicting_groups" and n_conflict_groups > 0:
            work = work[~work["_norm"].isin(conflicts["_norm"])].copy()

        before = len(work)
        sort_col = "date_parsed" if "date_parsed" in work.columns else "claim_id"
        work = work.sort_values(sort_col).drop_duplicates(subset=["_norm"], keep="first")
        after = len(work)
        print(f"drop_duplicate_statements ({policy}): {before} -> {after} (removed {before - after})")

        return work.drop(columns=["_norm"]).reset_index(drop=True)

    # ================================================================
    # Step 8: Time features (for Time2Vec / domain-adversarial component)
    # ================================================================

    def add_time_rank(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adds `time_rank` in [0, 1], a global linspace over the full
        chronologically-sorted frame. Must be called AFTER
        parse_and_sort_by_date(). Matches
        liar2-split-and-input-ablation-experiments3.ipynb and the main
        model notebook exactly (global rank, computed before any re-split)."""
        df = df.copy()
        df["time_rank"] = np.linspace(0.0, 1.0, len(df))
        return df

    def make_domain_buckets(self, time_rank_train: np.ndarray, num_domains: int = 5) -> np.ndarray:
        """Quantile-bucket TRAIN rows' time_rank into `num_domains` discrete
        ids, for domain-adversarial training. Pass only train-split
        time_rank values -- never valid/test -- to avoid leakage into the
        domain-bucket boundaries."""
        quantiles = np.quantile(time_rank_train, np.linspace(0, 1, num_domains + 1))
        quantiles[0] = -np.inf
        quantiles[-1] = np.inf
        return np.digitize(time_rank_train, quantiles[1:-1], right=True)

    # ================================================================
    # Re-splits (project's own 70/15/15 protocol, distinct from
    # official_split)
    # ================================================================

    def chronological_split(self, df: pd.DataFrame):
        """Contiguous 70/15/15 split in row order. Requires df to already
        be chronologically sorted (i.e. call parse_and_sort_by_date() first
        with sort_source == 'date')."""
        n = len(df)
        train_end = int(n * self.train_ratio)
        valid_end = int(n * (self.train_ratio + self.valid_ratio))
        return (
            np.arange(0, train_end),
            np.arange(train_end, valid_end),
            np.arange(valid_end, n),
        )

    def random_stratified_split(self, df: pd.DataFrame, seed: int = None):
        """70/15/15 stratified-by-label split, re-sorted chronologically
        within each split so recency weighting remains meaningful."""
        seed = self.seed if seed is None else seed
        all_idx = np.arange(len(df))
        y = df["label_id"].values

        train_idx, temp_idx = train_test_split(
            all_idx, train_size=self.train_ratio, random_state=seed, stratify=y,
        )
        y_temp = y[temp_idx]
        valid_relative_ratio = self.valid_ratio / (self.valid_ratio + self.test_ratio)
        valid_idx, test_idx = train_test_split(
            temp_idx, train_size=valid_relative_ratio, random_state=seed, stratify=y_temp,
        )
        return np.sort(train_idx), np.sort(valid_idx), np.sort(test_idx)

    def get_split(self, df: pd.DataFrame, split_mode: str, seed: int = None):
        """Added so Liar2Preprocessor is genuinely interchangeable with
        TemporalSplitter wherever a `.splitter` is expected (e.g.
        DAMiniLMExperimentRunner.run_full_battery()) -- confirmed missing
        via a real crash (AttributeError) when first wiring the two
        together in a full end-to-end run; not just a hypothetical gap."""
        if split_mode == "chronological":
            return self.chronological_split(df)
        if split_mode == "random_stratified":
            return self.random_stratified_split(df, seed=seed)
        raise ValueError(f"Unknown split_mode: {split_mode}")

    def make_rolling_windows(
        self,
        df: pd.DataFrame,
        min_train_frac: float = 0.50,
        valid_frac: float = 0.10,
        test_frac: float = 0.10,
        step_frac: float = 0.10,
    ) -> list:
        """Expanding-window temporal evaluation windows, for the temporal-
        generalization / rolling-window diagnostics."""
        n = len(df)
        windows = []
        train_end = int(n * min_train_frac)
        valid_size = int(n * valid_frac)
        test_size = int(n * test_frac)
        step = max(1, int(n * step_frac))

        window_id = 0
        while train_end + valid_size + test_size <= n:
            valid_end = train_end + valid_size
            test_end = valid_end + test_size
            train_idx = np.arange(0, train_end)
            valid_idx = np.arange(train_end, valid_end)
            test_idx = np.arange(valid_end, test_end)

            windows.append({
                "window_id": window_id,
                "train_idx": train_idx, "valid_idx": valid_idx, "test_idx": test_idx,
                # These three were missing until a real end-to-end dry run
                # crashed inside DAMiniLMExperimentRunner.run_rolling_window_evaluation(),
                # which expects the same keys TemporalSplitter.make_rolling_windows()
                # returns -- added here so the two are genuinely interchangeable,
                # not just interchangeable for chronological_split/random_stratified_split.
                "train_end_frac": train_end / n, "valid_end_frac": valid_end / n, "test_end_frac": test_end / n,
                "n_train": len(train_idx), "n_valid": len(valid_idx), "n_test": len(test_idx),
            })
            train_end += step
            window_id += 1
        return windows

    # ================================================================
    # Full pipeline (runs every step above, in order)
    # ================================================================

    def run_pipeline(self, train_path, valid_path, test_path) -> pd.DataFrame:
        """Runs every step in sequence:
            load_raw -> clean_statement_text -> normalize_labels ->
            filter_invalid_rows -> parse_and_sort_by_date ->
            build_text_modes -> drop_leakage_columns -> add_time_rank
        Each step prints its own diagnostic line, so a single call here
        gives you the full audit trail, not just the final DataFrame.
        """
        df = self.load_raw(train_path, valid_path, test_path)
        df = self.clean_statement_text(df)
        df = self.normalize_labels(df)
        df = self.filter_invalid_rows(df)
        df = self.parse_and_sort_by_date(df)
        df = self.build_text_modes(df)
        df = self.drop_leakage_columns(df)
        df = self.add_time_rank(df)

        print(f"\nrun_pipeline complete: {len(df)} final rows.")
        print("Official split counts (post-cleaning):")
        print(df["official_split"].value_counts().reindex(["train", "valid", "test"]))
        return df


if __name__ == "__main__":
    print("liar2_preprocessor.py loaded. Class: Liar2Preprocessor")

liar2_preprocessor.py loaded. Class: Liar2Preprocessor


In [7]:
"""
drift_analysis.py

Implements D01-D05 from d01-d05-drift-experiments.ipynb: label, vocabulary,
metadata, embedding, and domain-classifier drift diagnostics comparing
LIAR2's chronological train split against its chronological test split.

Uses Liar2Preprocessor for all data loading/cleaning -- this file only
computes drift statistics on top of what that class produces. To match
the original notebook's cohort exactly, the DataFrame passed in must be
deduplicated AND chronologically re-split first:

    from liar2_preprocessor import Liar2Preprocessor
    p = Liar2Preprocessor()
    df = p.run_pipeline(train_path, valid_path, test_path)
    audit = p.audit_duplicate_statements(df)
    df = p.drop_duplicate_statements(df, policy="keep_first", audit_df=audit)
    train_idx, valid_idx, test_idx = p.chronological_split(df)
    train_df, valid_df, test_df = df.iloc[train_idx], df.iloc[valid_idx], df.iloc[test_idx]

This exact sequence (dedup THEN chronological split) reproduces the
original notebook's reported cohort sizes (16,046 / 3,438 / 3,439) --
confirmed in an earlier step of this project. Skipping the dedup step
before splitting will NOT match the reported sample sizes.

One schema note: the original notebook's METADATA_COLUMNS list includes
`party_affiliation`, which does not exist in LIAR2's actual schema (it is
a LIAR-only field) -- confirmed against the real uploaded LIAR2 CSVs
(columns: id, label, statement, date, subject, speaker,
speaker_description, state_info, true_counts, ..., context,
justification). `run_metadata_drift()` here only uses columns that
genuinely exist on the LIAR2 DataFrame it's given (subject, speaker,
speaker_description, state_info, context, and optionally official_split
as a stand-in for the notebook's "original_split"), silently skipping
anything absent -- matching the original notebook's own
`available_metadata_cols` filtering behavior, not erroring on it.

D04 (MiniLM embedding shift) requires downloading
`microsoft/MiniLM-L12-H384-uncased` from the Hugging Face Hub, which this
sandbox's network cannot reach. It is written to match the notebook
exactly but NOT executed here -- run and verify it yourself before
trusting its output. D01, D02, D03, and D05 require no network access and
are tested end-to-end against real data in this project.

Usage
-----
    from drift_analysis import DriftAnalysis

    da = DriftAnalysis()
    d01 = da.run_label_drift(train_df, test_df)
    d02 = da.run_vocabulary_drift(train_df, test_df)
    d03 = da.run_metadata_drift(train_df, test_df)
    d05 = da.run_domain_classifier(train_df, test_df)

    # everything at once, plus a comparison against the originally
    # reported values:
    summary_df = da.run_all_drift_experiments(train_df, test_df, include_embedding_shift=False)
    da.compare_to_reported(summary_df)
"""

import re
from typing import Optional, Sequence

import numpy as np
import pandas as pd

from scipy.spatial.distance import jensenshannon
from scipy.stats import chi2_contingency

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity, rbf_kernel
from sklearn.decomposition import PCA


# Originally reported values from drift_shifting_experiment_results_liar2.xlsx
# ("Drift Metrics" sheet), for compare_to_reported().
REPORTED_VALUES = {
    "D01": {"js_distance": 0.396195, "total_variation": 0.428790},
    "D02": {"js_distance": 0.419836, "total_variation": 0.376209},
    "D03": {"speaker_js": 0.645526, "context_js": 0.597155},
    "D04": {"mmd_rbf": 0.025154, "centroid_cosine_distance": 0.003976},
    "D05": {"domain_auc": 0.908728, "domain_accuracy": 0.886418},
}


class DriftAnalysis:

    def __init__(
        self,
        label_order: Sequence[str] = None,
        random_state: int = 42,
        vocab_max_features: int = 30000,
        vocab_min_df: int = 2,
        top_shifted_terms: int = 40,
        top_metadata_categories: int = 50,
        embedding_model_name: str = "microsoft/MiniLM-L12-H384-uncased",
        embedding_sample_size_per_split: int = 2000,
        embedding_batch_size: int = 32,
        embedding_max_length: int = 128,
        domain_test_size: float = 0.30,
    ):
        self.label_order = label_order or [
            "pants-fire", "false", "barely-true", "half-true", "mostly-true", "true",
        ]
        self.random_state = random_state
        self.vocab_max_features = vocab_max_features
        self.vocab_min_df = vocab_min_df
        self.top_shifted_terms = top_shifted_terms
        self.top_metadata_categories = top_metadata_categories
        self.embedding_model_name = embedding_model_name
        self.embedding_sample_size_per_split = embedding_sample_size_per_split
        self.embedding_batch_size = embedding_batch_size
        self.embedding_max_length = embedding_max_length
        self.domain_test_size = domain_test_size

    # ================================================================
    # Shared utilities (match the notebook exactly)
    # ================================================================

    @staticmethod
    def normalize_text(text) -> str:
        text = str(text).lower().strip()
        return re.sub(r"\s+", " ", text)

    @staticmethod
    def safe_distribution(series: pd.Series, categories: Sequence[str]) -> np.ndarray:
        counts = series.value_counts()
        values = np.array([counts.get(c, 0) for c in categories], dtype=float)
        if values.sum() == 0:
            return np.ones(len(categories)) / len(categories)
        return values / values.sum()

    @staticmethod
    def js_distance(p, q) -> float:
        p = np.asarray(p, dtype=float)
        q = np.asarray(q, dtype=float)
        p = p / p.sum()
        q = q / q.sum()
        return float(jensenshannon(p, q, base=2))

    @staticmethod
    def total_variation_distance(p, q) -> float:
        p = np.asarray(p, dtype=float)
        q = np.asarray(q, dtype=float)
        p = p / p.sum()
        q = q / q.sum()
        return float(0.5 * np.abs(p - q).sum())

    @staticmethod
    def interpret_js_distance(value: float) -> str:
        if value < 0.05:
            return "very weak drift"
        elif value < 0.10:
            return "weak drift"
        elif value < 0.20:
            return "moderate drift"
        return "strong drift"

    @staticmethod
    def interpret_domain_auc(auc: float) -> str:
        if auc < 0.60:
            return "weak domain separability"
        elif auc < 0.75:
            return "moderate domain separability"
        elif auc < 0.90:
            return "strong domain separability"
        return "very strong domain separability"

    # ================================================================
    # D01: Label drift
    # ================================================================

    def run_label_drift(self, train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
        train_p = self.safe_distribution(train_df["label"], self.label_order)
        test_p = self.safe_distribution(test_df["label"], self.label_order)

        label_dist_df = pd.DataFrame({
            "label": self.label_order,
            "train_count": [train_df["label"].value_counts().get(l, 0) for l in self.label_order],
            "test_count": [test_df["label"].value_counts().get(l, 0) for l in self.label_order],
            "train_proportion": train_p,
            "test_proportion": test_p,
            "abs_train_test_difference": np.abs(train_p - test_p),
        })

        contingency = np.vstack([
            [train_df["label"].value_counts().get(l, 0) for l in self.label_order],
            [test_df["label"].value_counts().get(l, 0) for l in self.label_order],
        ])
        chi2, p_value, dof, expected = chi2_contingency(contingency)

        js = self.js_distance(train_p, test_p)
        summary = {
            "experiment_id": "D01",
            "analysis": "Label drift",
            "comparison": "chronological train vs test",
            "train_n": int(len(train_df)),
            "test_n": int(len(test_df)),
            "js_distance_train_test": js,
            "total_variation_train_test": self.total_variation_distance(train_p, test_p),
            "chi_square_train_test": float(chi2),
            "chi_square_p_value_train_test": float(p_value),
            "interpretation": self.interpret_js_distance(js),
        }
        return {"summary": summary, "label_distribution": label_dist_df}

    # ================================================================
    # D02: Vocabulary drift
    # ================================================================

    def run_vocabulary_drift(self, train_df: pd.DataFrame, test_df: pd.DataFrame, text_col: str = "statement") -> dict:
        train_texts = train_df[text_col].map(self.normalize_text).tolist()
        test_texts = test_df[text_col].map(self.normalize_text).tolist()

        vectorizer = CountVectorizer(
            lowercase=True, stop_words="english", ngram_range=(1, 2),
            max_features=self.vocab_max_features, min_df=self.vocab_min_df,
        )
        X_all = vectorizer.fit_transform(train_texts + test_texts)
        X_train = X_all[:len(train_texts)]
        X_test = X_all[len(train_texts):]

        train_counts = np.asarray(X_train.sum(axis=0)).ravel().astype(float)
        test_counts = np.asarray(X_test.sum(axis=0)).ravel().astype(float)

        # Add-one smoothing
        train_probs = (train_counts + 1.0)
        test_probs = (test_counts + 1.0)
        train_probs = train_probs / train_probs.sum()
        test_probs = test_probs / test_probs.sum()

        vocab = np.array(vectorizer.get_feature_names_out())
        difference = test_probs - train_probs

        vocab_df = pd.DataFrame({
            "term": vocab, "train_probability": train_probs, "test_probability": test_probs,
            "test_minus_train": difference, "absolute_difference": np.abs(difference),
        })
        terms_more_in_train = vocab_df.sort_values("test_minus_train", ascending=True).head(self.top_shifted_terms)
        terms_more_in_test = vocab_df.sort_values("test_minus_train", ascending=False).head(self.top_shifted_terms)

        js = self.js_distance(train_probs, test_probs)
        summary = {
            "experiment_id": "D02",
            "analysis": "Vocabulary drift",
            "comparison": "chronological train vs test",
            "train_n": int(len(train_df)),
            "test_n": int(len(test_df)),
            "n_vocabulary_terms": int(len(vocab)),
            "js_distance_train_test": js,
            "total_variation_train_test": self.total_variation_distance(train_probs, test_probs),
            "interpretation": self.interpret_js_distance(js),
        }
        return {
            "summary": summary, "vocab_df": vocab_df,
            "terms_more_in_train": terms_more_in_train, "terms_more_in_test": terms_more_in_test,
        }

    # ================================================================
    # D03: Metadata drift
    # ================================================================

    def _prepare_metadata_categories(self, train_series: pd.Series, test_series: pd.Series, top_k: int):
        combined = pd.concat([train_series, test_series], axis=0).astype(str).fillna("MISSING").str.strip()
        combined = combined.replace({"": "MISSING", "nan": "MISSING", "None": "MISSING", "none": "MISSING"})
        top_categories = combined.value_counts().head(top_k).index.tolist()

        train_clean = train_series.astype(str).fillna("MISSING").str.strip()
        test_clean = test_series.astype(str).fillna("MISSING").str.strip()
        train_clean = train_clean.where(train_clean.isin(top_categories), "OTHER")
        test_clean = test_clean.where(test_clean.isin(top_categories), "OTHER")

        return train_clean, test_clean, top_categories + ["OTHER"]

    def run_metadata_drift(
        self, train_df: pd.DataFrame, test_df: pd.DataFrame,
        metadata_columns: Optional[Sequence[str]] = None,
    ) -> dict:
        """Defaults to LIAR2's actual metadata columns (subject, speaker,
        speaker_description, state_info, context, official_split) --
        NOT the original notebook's list verbatim, which included
        `party_affiliation` (a LIAR-only field absent from real LIAR2) and
        `original_split` (renamed here to `official_split`, matching
        Liar2Preprocessor's column name). Any column not present on the
        given DataFrames is silently skipped, matching the notebook's own
        `available_metadata_cols` filtering behavior."""
        metadata_columns = metadata_columns or [
            "subject", "speaker", "speaker_description", "state_info", "context", "official_split",
        ]
        available_cols = [c for c in metadata_columns if c in train_df.columns and c in test_df.columns]
        if not available_cols:
            raise ValueError("No metadata columns found for D03.")

        summary_rows = []
        shifted_category_rows = []

        for col in available_cols:
            train_col, test_col, categories = self._prepare_metadata_categories(
                train_df[col], test_df[col], top_k=self.top_metadata_categories,
            )
            train_p = self.safe_distribution(train_col, categories)
            test_p = self.safe_distribution(test_col, categories)
            col_js = self.js_distance(train_p, test_p)
            col_tvd = self.total_variation_distance(train_p, test_p)

            contingency = np.vstack([
                [train_col.value_counts().get(c, 0) for c in categories],
                [test_col.value_counts().get(c, 0) for c in categories],
            ])
            try:
                chi2, p_value, dof, expected = chi2_contingency(contingency)
            except Exception:
                chi2, p_value = np.nan, np.nan

            summary_rows.append({
                "experiment_id": "D03", "analysis": "Metadata drift",
                "comparison": "chronological train vs test", "metadata_column": col,
                "n_categories_after_topk": len(categories),
                "js_distance_train_test": col_js, "total_variation_train_test": col_tvd,
                "chi_square_train_test": chi2, "chi_square_p_value_train_test": p_value,
                "interpretation": self.interpret_js_distance(col_js),
            })

            shifted_category_rows.append(pd.DataFrame({
                "metadata_column": col, "category": categories,
                "train_proportion": train_p, "test_proportion": test_p,
                "test_minus_train": test_p - train_p, "absolute_difference": np.abs(test_p - train_p),
            }).sort_values("absolute_difference", ascending=False))

        summary_df = pd.DataFrame(summary_rows).sort_values("js_distance_train_test", ascending=False)
        shifted_categories_df = pd.concat(shifted_category_rows, ignore_index=True)

        return {"summary_df": summary_df, "shifted_categories_df": shifted_categories_df}

    # ================================================================
    # D04: MiniLM embedding shift -- WRITTEN BUT NOT RUNNABLE IN THIS
    # SANDBOX (no network access to huggingface.co). Test before use.
    # ================================================================

    @staticmethod
    def _mean_pooling(last_hidden_state, attention_mask):
        import torch
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def _encode_texts_minilm(self, texts, tokenizer, model, device):
        import torch
        model.eval()
        all_embeddings = []
        with torch.no_grad():
            for start in range(0, len(texts), self.embedding_batch_size):
                batch_texts = texts[start:start + self.embedding_batch_size]
                enc = tokenizer(batch_texts, padding=True, truncation=True,
                                 max_length=self.embedding_max_length, return_tensors="pt")
                enc = {k: v.to(device) for k, v in enc.items()}
                outputs = model(**enc)
                emb = self._mean_pooling(outputs.last_hidden_state, enc["attention_mask"])
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
                all_embeddings.append(emb.cpu().numpy())
        return np.vstack(all_embeddings)

    def _stratified_sample_for_embeddings(self, df_part: pd.DataFrame, sample_size: int) -> pd.DataFrame:
        if len(df_part) <= sample_size:
            return df_part.copy()
        sampled_parts = []
        for label, group in df_part.groupby("label"):
            label_frac = len(group) / len(df_part)
            label_n = max(1, int(round(sample_size * label_frac)))
            sampled_parts.append(group.sample(n=min(label_n, len(group)), random_state=self.random_state))
        sampled = pd.concat(sampled_parts, ignore_index=True)
        if len(sampled) > sample_size:
            sampled = sampled.sample(n=sample_size, random_state=self.random_state)
        return sampled.reset_index(drop=True)

    def _compute_mmd_rbf(self, X: np.ndarray, Y: np.ndarray, max_mmd_samples: int = 1000):
        if len(X) > max_mmd_samples:
            rng = np.random.default_rng(self.random_state)
            X = X[rng.choice(len(X), max_mmd_samples, replace=False)]
        if len(Y) > max_mmd_samples:
            rng = np.random.default_rng(self.random_state)
            Y = Y[rng.choice(len(Y), max_mmd_samples, replace=False)]

        combined = np.vstack([X, Y])
        pairwise_sq = np.sum((combined[:, None, :] - combined[None, :, :]) ** 2, axis=-1)
        median_sq = np.median(pairwise_sq[pairwise_sq > 0])
        gamma = 1.0 if (median_sq <= 0 or np.isnan(median_sq)) else 1.0 / median_sq

        Kxx = rbf_kernel(X, X, gamma=gamma)
        Kyy = rbf_kernel(Y, Y, gamma=gamma)
        Kxy = rbf_kernel(X, Y, gamma=gamma)
        mmd = Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()
        return float(mmd), float(gamma)

    def run_embedding_shift(self, train_df: pd.DataFrame, test_df: pd.DataFrame, text_col: str = "statement") -> dict:
        """NOT TESTED in this sandbox -- requires downloading
        microsoft/MiniLM-L12-H384-uncased from the Hugging Face Hub, which
        this environment's network cannot reach. Verify in your own
        internet-enabled environment before trusting the output."""
        import torch
        from transformers import AutoTokenizer, AutoModel

        train_sample = self._stratified_sample_for_embeddings(train_df, self.embedding_sample_size_per_split)
        test_sample = self._stratified_sample_for_embeddings(test_df, self.embedding_sample_size_per_split)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.embedding_model_name)
        model = AutoModel.from_pretrained(self.embedding_model_name).to(device)

        train_emb = self._encode_texts_minilm(train_sample[text_col].tolist(), tokenizer, model, device)
        test_emb = self._encode_texts_minilm(test_sample[text_col].tolist(), tokenizer, model, device)

        train_centroid = train_emb.mean(axis=0, keepdims=True)
        test_centroid = test_emb.mean(axis=0, keepdims=True)
        centroid_cosine_similarity = cosine_similarity(train_centroid, test_centroid)[0, 0]
        centroid_cosine_distance = 1.0 - centroid_cosine_similarity

        mmd_rbf, mmd_gamma = self._compute_mmd_rbf(train_emb, test_emb)

        summary = {
            "experiment_id": "D04", "analysis": "MiniLM embedding shift",
            "comparison": "chronological train vs test", "embedding_model": self.embedding_model_name,
            "train_sample_size": int(len(train_sample)), "test_sample_size": int(len(test_sample)),
            "embedding_dim": int(train_emb.shape[1]),
            "centroid_cosine_similarity": float(centroid_cosine_similarity),
            "centroid_cosine_distance": float(centroid_cosine_distance),
            "mmd_rbf": float(mmd_rbf), "mmd_rbf_gamma": float(mmd_gamma),
            "interpretation": "larger centroid distance and MMD indicate stronger semantic embedding shift",
        }

        all_emb = np.vstack([train_emb, test_emb])
        split_labels = np.array(["train"] * len(train_emb) + ["test"] * len(test_emb))
        pca = PCA(n_components=2, random_state=self.random_state)
        pca_2d = pca.fit_transform(all_emb)
        pca_df = pd.DataFrame({"pc1": pca_2d[:, 0], "pc2": pca_2d[:, 1], "split": split_labels})

        return {"summary": summary, "pca_df": pca_df}

    # ================================================================
    # D05: Domain classifier drift
    # ================================================================

    def run_domain_classifier(self, train_df: pd.DataFrame, test_df: pd.DataFrame, text_col: str = "statement") -> dict:
        old_df = train_df[[text_col]].rename(columns={text_col: "text"}).copy()
        new_df = test_df[[text_col]].rename(columns={text_col: "text"}).copy()
        old_df["domain"] = 0
        new_df["domain"] = 1

        domain_df = pd.concat([old_df, new_df], ignore_index=True)
        domain_df = domain_df.sample(frac=1.0, random_state=self.random_state).reset_index(drop=True)

        X_train, X_eval, y_train, y_eval = train_test_split(
            domain_df["text"], domain_df["domain"], test_size=self.domain_test_size,
            random_state=self.random_state, stratify=domain_df["domain"],
        )

        domain_clf = Pipeline([
            ("tfidf", TfidfVectorizer(
                lowercase=True, strip_accents="unicode", stop_words="english",
                ngram_range=(1, 2), max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True,
            )),
            # solver="liblinear" is safe here (unlike the multiclass case
            # elsewhere in this project) -- this is a binary classification
            # problem (old domain vs new domain), and liblinear's binary
            # path was never affected by the multiclass-OvR removal that
            # broke the 6-class baselines on current scikit-learn.
            ("clf", LogisticRegression(solver="liblinear", C=1.0, max_iter=3000, random_state=self.random_state)),
        ])
        domain_clf.fit(X_train, y_train)

        y_pred = domain_clf.predict(X_eval)
        y_score = domain_clf.predict_proba(X_eval)[:, 1]

        acc = accuracy_score(y_eval, y_pred)
        f1 = f1_score(y_eval, y_pred)
        auc = roc_auc_score(y_eval, y_score)

        summary = {
            "experiment_id": "D05", "analysis": "Domain classifier drift",
            "comparison": "chronological train vs test",
            "domain_0": "old chronological train", "domain_1": "new chronological test",
            "n_total": int(len(domain_df)), "n_eval": int(len(X_eval)),
            "domain_accuracy": float(acc), "domain_f1": float(f1), "domain_auc": float(auc),
            "interpretation": self.interpret_domain_auc(auc),
        }

        report_str = classification_report(
            y_eval, y_pred, target_names=["old_train_domain", "new_test_domain"], digits=4,
        )

        vectorizer = domain_clf.named_steps["tfidf"]
        classifier = domain_clf.named_steps["clf"]
        feature_names = np.array(vectorizer.get_feature_names_out())
        coefs = classifier.coef_[0]
        top_new_idx = np.argsort(coefs)[-40:][::-1]
        top_old_idx = np.argsort(coefs)[:40]
        top_domain_terms = pd.concat([
            pd.DataFrame({"term": feature_names[top_new_idx], "coefficient": coefs[top_new_idx],
                          "more_associated_with": "new_test_domain"}),
            pd.DataFrame({"term": feature_names[top_old_idx], "coefficient": coefs[top_old_idx],
                          "more_associated_with": "old_train_domain"}),
        ], ignore_index=True)

        return {"summary": summary, "classification_report": report_str, "top_domain_terms": top_domain_terms}

    # ================================================================
    # Convenience: run everything, plus comparison against reported values
    # ================================================================

    def run_all_drift_experiments(
        self, train_df: pd.DataFrame, test_df: pd.DataFrame,
        metadata_columns: Optional[Sequence[str]] = None,
        include_embedding_shift: bool = False,
    ) -> pd.DataFrame:
        """Runs D01, D02, D03, D05 (and D04 if include_embedding_shift=True
        -- requires Hugging Face Hub access). Returns a DataFrame with one
        row per D0x summary (D03 contributes one row per metadata column)."""
        rows = []

        print("running D01: Label drift")
        rows.append(self.run_label_drift(train_df, test_df)["summary"])

        print("running D02: Vocabulary drift")
        rows.append(self.run_vocabulary_drift(train_df, test_df)["summary"])

        print("running D03: Metadata drift")
        d03 = self.run_metadata_drift(train_df, test_df, metadata_columns=metadata_columns)
        rows.extend(d03["summary_df"].to_dict("records"))

        if include_embedding_shift:
            print("running D04: MiniLM embedding shift")
            rows.append(self.run_embedding_shift(train_df, test_df)["summary"])

        print("running D05: Domain classifier drift")
        rows.append(self.run_domain_classifier(train_df, test_df)["summary"])

        print("\nAll drift experiments complete.")
        return pd.DataFrame(rows)

    def compare_to_reported(self, summary_df: pd.DataFrame) -> pd.DataFrame:
        """Compares this run's D01/D02/D05 metrics (and D03's speaker/
        context JS if present) against REPORTED_VALUES, the figures from
        drift_shifting_experiment_results_liar2.xlsx's "Drift Metrics"
        sheet. Returns a small comparison table; also prints it."""
        rows = []

        d01 = summary_df[summary_df["experiment_id"] == "D01"]
        if len(d01):
            r = d01.iloc[0]
            rows.append(("D01 JS distance", r["js_distance_train_test"], REPORTED_VALUES["D01"]["js_distance"]))
            rows.append(("D01 total variation", r["total_variation_train_test"], REPORTED_VALUES["D01"]["total_variation"]))

        d02 = summary_df[summary_df["experiment_id"] == "D02"]
        if len(d02):
            r = d02.iloc[0]
            rows.append(("D02 JS distance", r["js_distance_train_test"], REPORTED_VALUES["D02"]["js_distance"]))
            rows.append(("D02 total variation", r["total_variation_train_test"], REPORTED_VALUES["D02"]["total_variation"]))

        d03 = summary_df[summary_df["experiment_id"] == "D03"]
        if len(d03) and "metadata_column" in d03.columns:
            speaker_row = d03[d03["metadata_column"] == "speaker"]
            if len(speaker_row):
                rows.append(("D03 speaker JS", speaker_row.iloc[0]["js_distance_train_test"],
                             REPORTED_VALUES["D03"]["speaker_js"]))
            context_row = d03[d03["metadata_column"] == "context"]
            if len(context_row):
                rows.append(("D03 context JS", context_row.iloc[0]["js_distance_train_test"],
                             REPORTED_VALUES["D03"]["context_js"]))

        d04 = summary_df[summary_df["experiment_id"] == "D04"]
        if len(d04):
            r = d04.iloc[0]
            rows.append(("D04 MMD (RBF)", r["mmd_rbf"], REPORTED_VALUES["D04"]["mmd_rbf"]))
            rows.append(("D04 centroid cosine distance", r["centroid_cosine_distance"],
                         REPORTED_VALUES["D04"]["centroid_cosine_distance"]))

        d05 = summary_df[summary_df["experiment_id"] == "D05"]
        if len(d05):
            r = d05.iloc[0]
            rows.append(("D05 domain AUC", r["domain_auc"], REPORTED_VALUES["D05"]["domain_auc"]))
            rows.append(("D05 domain accuracy", r["domain_accuracy"], REPORTED_VALUES["D05"]["domain_accuracy"]))

        comparison_df = pd.DataFrame(rows, columns=["metric", "this_run", "originally_reported"])
        comparison_df["abs_difference"] = (comparison_df["this_run"] - comparison_df["originally_reported"]).abs()

        print(comparison_df.to_string(index=False))
        return comparison_df


if __name__ == "__main__":
    print("drift_analysis.py loaded. Class: DriftAnalysis "
          "(methods: run_label_drift, run_vocabulary_drift, run_metadata_drift, "
          "run_embedding_shift, run_domain_classifier, run_all_drift_experiments, "
          "compare_to_reported)")

drift_analysis.py loaded. Class: DriftAnalysis (methods: run_label_drift, run_vocabulary_drift, run_metadata_drift, run_embedding_shift, run_domain_classifier, run_all_drift_experiments, compare_to_reported)


# temporal_splitter

In [8]:
"""
temporal_splitter.py

Class 1 of 3: splitting logic only, from da-minilm-main-model-v1-0.ipynb's
cell 4 ("Split functions: random, chronological, and rolling windows").

This is a standalone, dataset-agnostic version of the same splitting
functions Liar2Preprocessor already exposes (chronological_split,
random_stratified_split, make_rolling_windows) -- pulled out on its own
here because the model class and the experiment-runner class in this
same three-class set shouldn't need to know about Liar2Preprocessor
specifically. Any DataFrame with a `label_id` integer column works.

If you're already using Liar2Preprocessor for LIAR2 specifically, its
built-in chronological_split()/random_stratified_split()/
make_rolling_windows() are equivalent to this class's methods (same
logic, verified against the same notebook) -- use whichever is more
convenient for your call site. DAMiniLMExperimentRunner (class 3) accepts
either.

Usage
-----
    from temporal_splitter import TemporalSplitter

    splitter = TemporalSplitter()
    train_idx, valid_idx, test_idx = splitter.chronological_split(df)
    train_idx, valid_idx, test_idx = splitter.random_stratified_split(df, seed=42)
    windows = splitter.make_rolling_windows(df)
"""

from typing import Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


class TemporalSplitter:

    def __init__(self, train_ratio: float = 0.70, valid_ratio: float = 0.15, test_ratio: float = 0.15,
                 seed: int = 42):
        assert abs((train_ratio + valid_ratio + test_ratio) - 1.0) < 1e-9, \
            "train_ratio + valid_ratio + test_ratio must sum to 1.0"
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio
        self.seed = seed

    def chronological_split(self, df: pd.DataFrame):
        """Contiguous 70/15/15 split in row order. `df` must already be
        chronologically sorted (e.g. by Liar2Preprocessor's
        parse_and_sort_by_date(), or any equivalent date-ordering step)."""
        n = len(df)
        train_end = int(n * self.train_ratio)
        valid_end = int(n * (self.train_ratio + self.valid_ratio))
        return np.arange(0, train_end), np.arange(train_end, valid_end), np.arange(valid_end, n)

    def random_stratified_split(self, df: pd.DataFrame, seed: Optional[int] = None):
        """70/15/15 stratified-by-label split (label taken from `label_id`)."""
        seed = self.seed if seed is None else seed
        all_idx = np.arange(len(df))
        y = df["label_id"].values

        train_idx, temp_idx = train_test_split(
            all_idx, train_size=self.train_ratio, random_state=seed, stratify=y,
        )
        y_temp = y[temp_idx]
        valid_relative_ratio = self.valid_ratio / (self.valid_ratio + self.test_ratio)
        valid_idx, test_idx = train_test_split(
            temp_idx, train_size=valid_relative_ratio, random_state=seed, stratify=y_temp,
        )
        return np.sort(train_idx), np.sort(valid_idx), np.sort(test_idx)

    def get_split(self, df: pd.DataFrame, split_mode: str, seed: Optional[int] = None):
        if split_mode == "chronological":
            return self.chronological_split(df)
        if split_mode == "random_stratified":
            return self.random_stratified_split(df, seed=seed)
        raise ValueError(f"Unknown split_mode: {split_mode}")

    def make_rolling_windows(
        self, df: pd.DataFrame, min_train_frac: float = 0.50, valid_frac: float = 0.10,
        test_frac: float = 0.10, step_frac: float = 0.10,
    ) -> list:
        """Expanding-window temporal evaluation. `df` must already be
        chronologically sorted. Example window: train = earliest 50%,
        valid = next 10%, test = next 10%; the training window then
        expands by step_frac each iteration."""
        n = len(df)
        windows = []
        train_end = int(n * min_train_frac)
        valid_size = int(n * valid_frac)
        test_size = int(n * test_frac)
        step = max(1, int(n * step_frac))

        window_id = 0
        while train_end + valid_size + test_size <= n:
            valid_end = train_end + valid_size
            test_end = valid_end + test_size

            train_idx = np.arange(0, train_end)
            valid_idx = np.arange(train_end, valid_end)
            test_idx = np.arange(valid_end, test_end)

            windows.append({
                "window_id": window_id, "train_idx": train_idx, "valid_idx": valid_idx, "test_idx": test_idx,
                "train_end_frac": train_end / n, "valid_end_frac": valid_end / n, "test_end_frac": test_end / n,
                "n_train": len(train_idx), "n_valid": len(valid_idx), "n_test": len(test_idx),
            })
            train_end += step
            window_id += 1

        return windows


if __name__ == "__main__":
    print("temporal_splitter.py loaded. Class: TemporalSplitter")

temporal_splitter.py loaded. Class: TemporalSplitter


# da_minilm_rwc_model

In [9]:
"""
da_minilm_rwc_model.py

Class 2 of 3: the model(s) only -- frozen MiniLM sentence embeddings, the
DA-MiniLM-r MLP classifier, and the frozen-embedding Logistic Regression
comparator (MiniLM-LogReg-RW). No splitting logic (see temporal_splitter
.py) and no experiment orchestration -- sample weighting, calibration,
multi-seed batteries, and bootstrap comparison (see
da_minilm_experiment_runner.py).

This class trains and predicts. It does not decide what data to train on,
how to weight it, or how to calibrate the output -- those are the
experiment runner's job, so this class stays reusable for other weighting
or calibration schemes without modification.
"""

import random
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


NUM_CLASSES_DEFAULT = 6


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class EmbeddingDataset(Dataset):
    def __init__(self, X, y=None, sample_weight=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
        if sample_weight is None:
            sample_weight = np.ones(len(X), dtype=np.float32)
        self.sample_weight = torch.tensor(sample_weight, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        item = {"x": self.X[idx], "sample_weight": self.sample_weight[idx]}
        if self.y is not None:
            item["y"] = self.y[idx]
        return item


class DAMiniLMClassifier(nn.Module):
    """The MLP head trained on frozen MiniLM embeddings."""

    def __init__(self, input_dim: int, num_classes: int = NUM_CLASSES_DEFAULT, hidden_dim: int = 256, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def predict_da_minilm_proba(model, loader, device) -> np.ndarray:
    model.eval()
    probas = []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            logits = model(x)
            probas.append(F.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(probas, axis=0)


class DAMiniLMModel:

    def __init__(
        self,
        output_dir="./da_minilm_results",
        minilm_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        device: Optional[str] = None,
        num_classes: int = NUM_CLASSES_DEFAULT,
        hidden_dim: int = 256,
        dropout: float = 0.25,
        mlp_epochs: int = 35,
        mlp_batch_size: int = 128,
        mlp_lr: float = 1e-3,
        mlp_weight_decay: float = 1e-4,
        mlp_patience: int = 6,
    ):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.minilm_model_name = minilm_model_name
        self.device = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.mlp_epochs = mlp_epochs
        self.mlp_batch_size = mlp_batch_size
        self.mlp_lr = mlp_lr
        self.mlp_weight_decay = mlp_weight_decay
        self.mlp_patience = mlp_patience

        self._embedding_cache = {}

    # ---------------- Embedding extraction ----------------

    def encode_texts_minilm(self, texts: Sequence[str], cache_key: str, batch_size: int = 128) -> np.ndarray:
        """Encode texts once, cached in memory and on disk. Requires
        downloading `self.minilm_model_name` from the Hugging Face Hub the
        first time it's called for a given environment."""
        cache_path = self.output_dir / f"embeddings_{cache_key}.npy"
        if cache_key in self._embedding_cache:
            return self._embedding_cache[cache_key]
        if cache_path.exists():
            X = np.load(cache_path)
            self._embedding_cache[cache_key] = X
            print(f"Loaded cached embeddings: {cache_path} {X.shape}")
            return X

        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(self.minilm_model_name, device=str(self.device))
        X = model.encode(
            list(texts), batch_size=batch_size, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)

        np.save(cache_path, X)
        self._embedding_cache[cache_key] = X
        print(f"Saved embeddings: {cache_path} {X.shape}")
        return X

    # ---------------- DA-MiniLM MLP ----------------

    def train_da_minilm_mlp(self, X_train, y_train, w_train, X_valid, y_valid, X_test, seed: int = 42):
        """Trains the MLP head on frozen embeddings with per-sample
        weights (`w_train`), early-stopping on validation macro-F1.
        Returns (valid_proba, test_proba, best_valid_macro_f1) -- raw,
        uncalibrated probabilities; calibration is the experiment
        runner's job, not this class's."""
        set_all_seeds(seed)

        train_ds = EmbeddingDataset(X_train, y_train, w_train)
        valid_ds = EmbeddingDataset(X_valid, y_valid)
        test_ds = EmbeddingDataset(X_test, None)

        generator = torch.Generator()
        generator.manual_seed(seed)

        train_loader = DataLoader(train_ds, batch_size=self.mlp_batch_size, shuffle=True, generator=generator)
        valid_loader = DataLoader(valid_ds, batch_size=self.mlp_batch_size, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=self.mlp_batch_size, shuffle=False)

        model = DAMiniLMClassifier(
            input_dim=X_train.shape[1], num_classes=self.num_classes,
            hidden_dim=self.hidden_dim, dropout=self.dropout,
        ).to(self.device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.mlp_lr, weight_decay=self.mlp_weight_decay)

        best_state = None
        best_valid_macro_f1 = -1.0
        bad_epochs = 0

        for epoch in range(self.mlp_epochs):
            model.train()
            for batch in train_loader:
                x = batch["x"].to(self.device)
                y = batch["y"].to(self.device)
                sw = batch["sample_weight"].to(self.device)

                logits = model(x)
                per_sample_loss = F.cross_entropy(logits, y, reduction="none")
                loss = (per_sample_loss * sw).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            valid_proba = predict_da_minilm_proba(model, valid_loader, self.device)
            valid_pred = valid_proba.argmax(axis=1)
            valid_macro_f1 = f1_score(y_valid, valid_pred, average="macro", zero_division=0)

            if valid_macro_f1 > best_valid_macro_f1:
                best_valid_macro_f1 = valid_macro_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
            if bad_epochs >= self.mlp_patience:
                break

        if best_state is not None:
            model.load_state_dict(best_state)

        valid_proba = predict_da_minilm_proba(model, valid_loader, self.device)
        test_proba = predict_da_minilm_proba(model, test_loader, self.device)
        return valid_proba, test_proba, best_valid_macro_f1

    # ---------------- Frozen-embedding Logistic Regression comparator ----------------

    def train_logreg_baseline(self, X_train, y_train, w_train, X_valid, y_valid, X_test, seed: int = 42):
        """MiniLM-LogReg-RW: frozen MiniLM embeddings + Logistic
        Regression. class_weight=None because sample_weight (w_train)
        already encodes class balance if the caller included it (see
        da_minilm_experiment_runner.py's combined_sample_weights)."""
        clf = make_pipeline(
            StandardScaler(with_mean=True),
            LogisticRegression(max_iter=2000, class_weight=None, random_state=seed),
        )
        clf.fit(X_train, y_train, logisticregression__sample_weight=w_train)
        return clf.predict_proba(X_valid), clf.predict_proba(X_test)


if __name__ == "__main__":
    print("da_minilm_model.py loaded. Class: DAMiniLMModel")

da_minilm_model.py loaded. Class: DAMiniLMModel


In [10]:
"""
da_minilm_experiment_runner.py

Class 3 of 3: experiment orchestration. Uses TemporalSplitter (class 1)
for splitting and DAMiniLMModel (class 2) for training/prediction -- this
class owns everything in between and around those two: metrics, sample
weighting (class-balance + recency), calibration (bias + temperature
selection), the recency-lambda grid search, multi-seed batteries,
result summarization, paired bootstrap comparison, and rolling-window
temporal evaluation.

--------------------------------------------------------------------------
A known, deliberately-preserved bug: the calibration temperature grid
--------------------------------------------------------------------------
select_best_calibration() grid-searches (tau, temperature) and keeps
whichever combination gets the best VALIDATION macro-F1. But temperature
scaling divides every class's logit by the same positive scalar -- a
monotonic transform that CANNOT change which class has the highest logit,
i.e. it cannot change argmax, i.e. it cannot change macro-F1 at all. For
any fixed tau, every temperature in temperature_grid produces the exact
same macro-F1. Because the selection loop uses strict `>` comparison,
the FIRST temperature value in the grid for the winning tau is silently
kept -- not because it's better, but because later ties never satisfy
`score > best["score"]`. This matches this project's own documented
finding ("temperature non-identifiable under macro-F1 objective; locks to
first grid value via iteration-order artifact") -- verified directly
(temperature confirmed to have zero effect on macro-F1 across the full
grid, and the first grid value confirmed to silently win). Reproduced
here byte-for-byte, commented rather than silently fixed, per this
project's practice of disclosing known bugs rather than patching them out
after the fact. Temperature DOES still matter for ECE/Brier/NLL (those
depend on the actual probabilities, not just argmax) -- it only fails to
matter for the objective this selection loop actually uses by default.
Pass objective="nll" to select_best_calibration() for an objective
temperature actually affects.
--------------------------------------------------------------------------

Usage
-----
    from temporal_splitter import TemporalSplitter
    from da_minilm_model import DAMiniLMModel
    from da_minilm_experiment_runner import DAMiniLMExperimentRunner

    splitter = TemporalSplitter()
    model = DAMiniLMModel(output_dir="./da_minilm_results")
    runner = DAMiniLMExperimentRunner(splitter=splitter, model=model)

    embeddings = {mode: model.encode_texts_minilm(df[f"model_text__{mode}"].tolist(), cache_key=mode)
                  for mode in ["claim_only", "claim_plus_safe_metadata"]}

    main_results_df, lambda_selection_df, prediction_store = runner.run_full_battery(
        df, embeddings, seeds=[42, 43, 44, 45, 46],
        text_modes=["claim_only", "claim_plus_safe_metadata"],
        split_modes=["chronological", "random_stratified"],
    )

    summary_df = runner.summarize_results(main_results_df, group_cols=["model", "text_mode", "split_mode"])
    bootstrap_df = runner.run_paired_bootstrap(prediction_store,
                                                text_modes=["claim_only", "claim_plus_safe_metadata"],
                                                split_modes=["chronological", "random_stratified"],
                                                seeds=[42, 43, 44, 45, 46])
"""

from typing import Optional, Sequence

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, log_loss,
)


NUM_CLASSES_DEFAULT = 6


# ============================================================
# Metrics
# ============================================================

def ordinal_mae(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def within_one_accuracy(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) <= 1))


def expected_calibration_error(y_true, proba, n_bins: int = 10) -> float:
    """Multiclass ECE based on confidence of the predicted class."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    ece = 0.0
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        bin_acc = correct[mask].mean()
        bin_conf = conf[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)
    return float(ece)


def multiclass_brier_score(y_true, proba, num_classes: int = NUM_CLASSES_DEFAULT) -> float:
    y_true = np.asarray(y_true)
    y_onehot = np.eye(num_classes)[y_true]
    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))


def safe_nll(y_true, proba, num_classes: int = NUM_CLASSES_DEFAULT) -> float:
    return float(log_loss(y_true, np.clip(proba, 1e-12, 1.0), labels=list(range(num_classes))))


def evaluate_predictions(
    y_true, y_pred, proba=None, model_name="model", split_mode=None, text_mode=None,
    seed=None, recency_lambda=None, tau=None, temperature=None, extra=None,
    num_classes: int = NUM_CLASSES_DEFAULT,
) -> dict:
    """`num_classes` MUST match proba.shape[1] -- it defaults to 6 (LIAR2)
    for backward compatibility, but this default silently breaks on any
    other label space (e.g. MultiFC's 3 classes) via a shape mismatch in
    multiclass_brier_score/safe_nll. Always pass num_classes explicitly
    when working outside the 6-class LIAR2/LIAR label space -- confirmed
    this default was a real bug when first reusing this code for MultiFC."""
    row = {
        "model": model_name, "split_mode": split_mode, "text_mode": text_mode, "seed": seed,
        "recency_lambda": recency_lambda, "tau": tau, "temperature": temperature,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "ordinal_mae": ordinal_mae(y_true, y_pred),
        "within_one_accuracy": within_one_accuracy(y_true, y_pred),
    }
    if proba is not None:
        row.update({
            "ece": expected_calibration_error(y_true, proba),
            "brier": multiclass_brier_score(y_true, proba, num_classes=num_classes),
            "nll": safe_nll(y_true, proba, num_classes=num_classes),
        })
    if extra:
        row.update(extra)
    return row


# ============================================================
# Sample weighting
# ============================================================

def normalize_weights(w, min_weight: float = 0.05, max_weight: float = 20.0) -> np.ndarray:
    w = np.asarray(w, dtype=float)
    w = np.clip(w, min_weight, max_weight)
    return w / (np.mean(w) + 1e-12)


def effective_number_class_weights(y, beta: float = 0.999, num_classes: int = NUM_CLASSES_DEFAULT) -> np.ndarray:
    """Per-class weights based on the effective number of samples (Cui et al., 2019)."""
    counts = np.bincount(np.asarray(y, dtype=int), minlength=num_classes).astype(float)
    weights = np.ones(num_classes, dtype=float)
    for c in range(num_classes):
        n = counts[c]
        if n > 0:
            effective_num = (1.0 - beta ** n) / (1.0 - beta)
            weights[c] = 1.0 / effective_num
    return weights / (weights.mean() + 1e-12)


def class_sample_weights(y_train, beta: float = 0.999, num_classes: int = NUM_CLASSES_DEFAULT) -> np.ndarray:
    class_w = effective_number_class_weights(y_train, beta=beta, num_classes=num_classes)
    return normalize_weights(class_w[np.asarray(y_train, dtype=int)])


def recency_sample_weights(time_rank_train, lam: float = 0.0) -> np.ndarray:
    """Exponential recency weighting: age = max_train_time - sample_time;
    higher lambda gives more weight to recent training samples."""
    t = np.asarray(time_rank_train, dtype=float)
    age = t.max() - t
    w = np.exp(-float(lam) * age * 10.0)
    return normalize_weights(w)


def combined_sample_weights(y_train, time_rank_train, recency_lambda: float = 0.0,
                             beta: float = 0.999, num_classes: int = NUM_CLASSES_DEFAULT) -> np.ndarray:
    w_class = class_sample_weights(y_train, beta=beta, num_classes=num_classes)
    w_recency = recency_sample_weights(time_rank_train, recency_lambda)
    return normalize_weights(w_class * w_recency)


# ============================================================
# Calibration
# ============================================================

def softmax_np(logits) -> np.ndarray:
    logits = np.asarray(logits, dtype=float)
    logits = logits - logits.max(axis=1, keepdims=True)
    expv = np.exp(logits)
    return expv / (expv.sum(axis=1, keepdims=True) + 1e-12)


def proba_to_logits(proba, eps: float = 1e-12) -> np.ndarray:
    return np.log(np.clip(proba, eps, 1.0))


def logit_adjustment_bias(y_train, num_classes: int = NUM_CLASSES_DEFAULT, tau: float = 1.0, eps: float = 1e-12) -> np.ndarray:
    counts = np.bincount(y_train, minlength=num_classes).astype(float)
    prior = counts / counts.sum()
    return tau * np.log(prior + eps)


def apply_bias_and_temperature(proba, bias=None, temperature: float = 1.0) -> np.ndarray:
    logits = proba_to_logits(proba)
    if bias is not None:
        logits = logits - bias.reshape(1, -1)
    logits = logits / max(float(temperature), 1e-6)
    return softmax_np(logits)


def select_best_calibration(
    valid_proba, y_valid, y_train, objective: str = "macro_f1",
    tau_grid: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
    temperature_grid: Sequence[float] = (0.75, 1.0, 1.25, 1.5, 2.0),
    num_classes: int = NUM_CLASSES_DEFAULT,
) -> dict:
    """Select tau and temperature using validation only.

    NOTE: with objective="macro_f1" (the default), temperature has no
    effect on the selected argmax and is therefore effectively
    unidentifiable -- see module docstring. Use objective="nll" for an
    objective temperature actually affects, or use
    select_best_calibration_two_stage() below instead of this function.
    """
    best = {"tau": 0.0, "temperature": 1.0, "score": -1e9}
    for tau in tau_grid:
        bias = logit_adjustment_bias(y_train, tau=tau, num_classes=num_classes)
        for temp in temperature_grid:
            calibrated = apply_bias_and_temperature(valid_proba, bias=bias, temperature=temp)
            pred = calibrated.argmax(axis=1)
            if objective == "macro_f1":
                score = f1_score(y_valid, pred, average="macro", zero_division=0)
            elif objective == "nll":
                score = -safe_nll(y_valid, calibrated, num_classes=num_classes)
            else:
                raise ValueError(f"Unknown objective: {objective}")
            if score > best["score"]:
                best = {"tau": tau, "temperature": temp, "score": float(score)}
    return best


def select_best_calibration_two_stage(
    valid_proba, y_valid, y_train,
    tau_grid: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
    temperature_grid: Sequence[float] = (0.75, 1.0, 1.25, 1.5, 2.0),
    num_classes: int = NUM_CLASSES_DEFAULT,
) -> dict:
    """The calibration-bug fix. select_best_calibration() jointly grid-
    searches (tau, temperature) under a single objective -- but temperature
    has zero effect on argmax, so under objective="macro_f1" it can never
    be meaningfully selected (see module docstring; confirmed by direct
    test). This function decouples the two selections onto the objective
    each one actually affects:

    Stage 1 -- select tau under macro_f1 (tau/logit-adjustment DOES shift
               the decision boundary via the class-prior bias, so this
               objective is meaningful here).
    Stage 2 -- with tau fixed at its stage-1 value, select temperature
               under NLL (temperature doesn't move argmax but does reshape
               the probability distribution, so NLL -- which depends on
               the full distribution, not just the top class -- is
               sensitive to it. This is the "objective each dimension can
               actually see" pairing that was missing from the joint
               search.)

    Returns {"tau", "temperature", "macro_f1_at_best_tau", "nll_at_best_temperature"}.
    """
    # Stage 1: tau, evaluated at temperature=1.0 (irrelevant to macro_f1 anyway).
    best_tau, best_tau_score = 0.0, -1e9
    for tau in tau_grid:
        bias = logit_adjustment_bias(y_train, tau=tau, num_classes=num_classes)
        calibrated = apply_bias_and_temperature(valid_proba, bias=bias, temperature=1.0)
        score = f1_score(y_valid, calibrated.argmax(axis=1), average="macro", zero_division=0)
        if score > best_tau_score:
            best_tau, best_tau_score = tau, float(score)

    # Stage 2: temperature, with tau fixed, evaluated under NLL.
    bias = logit_adjustment_bias(y_train, tau=best_tau, num_classes=num_classes)
    best_temp, best_temp_score = 1.0, 1e9
    for temp in temperature_grid:
        calibrated = apply_bias_and_temperature(valid_proba, bias=bias, temperature=temp)
        nll = safe_nll(y_valid, calibrated, num_classes=num_classes)
        if nll < best_temp_score:
            best_temp, best_temp_score = temp, float(nll)

    return {
        "tau": best_tau, "temperature": best_temp,
        "macro_f1_at_best_tau": best_tau_score, "nll_at_best_temperature": best_temp_score,
    }


# ============================================================
# Statistical comparison
# ============================================================

def paired_bootstrap_macro_f1_diff(y_true, y_pred_a, y_pred_b, n_boot: int = 2000, seed: int = 42) -> dict:
    rng = np.random.RandomState(seed)
    y_true, y_pred_a, y_pred_b = np.asarray(y_true), np.asarray(y_pred_a), np.asarray(y_pred_b)
    n = len(y_true)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        f1_a = f1_score(y_true[idx], y_pred_a[idx], average="macro", zero_division=0)
        f1_b = f1_score(y_true[idx], y_pred_b[idx], average="macro", zero_division=0)
        diffs[b] = f1_a - f1_b

    ci_lower, ci_upper = np.percentile(diffs, [2.5, 97.5])
    prop_le_0 = float(np.mean(diffs <= 0))
    prop_ge_0 = float(np.mean(diffs >= 0))
    p_value = min(2.0 * min(prop_le_0, prop_ge_0), 1.0)

    return {
        "mean_diff": float(diffs.mean()), "ci_lower_95": float(ci_lower), "ci_upper_95": float(ci_upper),
        "p_value": p_value, "significant_at_95": bool(ci_lower > 0 or ci_upper < 0),
    }


# ============================================================
# Experiment runner
# ============================================================

class DAMiniLMExperimentRunner:

    def __init__(
        self,
        splitter,
        model,
        recency_lambdas: Sequence[float] = (0.0, 0.05, 0.10, 0.20, 0.30),
        class_weight_beta: float = 0.999,
        temperature_grid: Sequence[float] = (0.75, 1.0, 1.25, 1.5, 2.0),
        tau_grid: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
        num_classes: int = NUM_CLASSES_DEFAULT,
        output_dir=None,
    ):
        """`splitter` should be a TemporalSplitter instance (class 1) --
        used for chronological_split / random_stratified_split /
        make_rolling_windows. `model` should be a DAMiniLMModel instance
        (class 2) -- used for train_da_minilm_mlp / train_logreg_baseline.
        Neither is reimplemented here."""
        self.splitter = splitter
        self.model = model
        self.recency_lambdas = recency_lambdas
        self.class_weight_beta = class_weight_beta
        self.temperature_grid = temperature_grid
        self.tau_grid = tau_grid
        self.num_classes = num_classes
        self.output_dir = output_dir or getattr(model, "output_dir", None)

    # ---------------- Single-run orchestration ----------------

    def run_single_experiment(
        self, X, y, time_rank, train_idx, valid_idx, test_idx,
        text_mode: str, split_mode: str, seed: int, model_kind: str = "da_minilm_mlp",
        use_reweighting: bool = True, use_calibration: bool = True,
        calibration_mode: str = "joint_macro_f1", model_name_override: Optional[str] = None,
    ):
        """model_kind: "da_minilm_mlp" (MLP head) or "minilm_logreg"
        (Logistic Regression head) -- the architecture dimension of the
        component ablation.

        use_reweighting: if False, sample weights are uniform (no class
        balancing, no recency weighting) and self.recency_lambdas is
        skipped entirely -- this is the ablation's "Uniform" condition
        for whichever architecture is selected (e.g. model_kind=
        "minilm_logreg" + use_reweighting=False == Uniform-LR).

        use_calibration: if False, raw (uncalibrated) argmax predictions
        are used directly -- no tau/temperature search at all. This
        isolates the calibration step's contribution.

        calibration_mode: only relevant when use_calibration=True.
            "joint_macro_f1" -- the ORIGINAL notebook's approach: joint
                (tau, temperature) grid search under macro_f1. Preserved
                for exact reproduction; see module docstring re: the
                temperature non-identifiability this causes.
            "two_stage" -- the BUG FIX: tau selected under macro_f1,
                temperature separately selected under NLL. Use this for
                new results; use "joint_macro_f1" only to reproduce prior
                numbers exactly.

        Returns (row, lambda_df, pred_test, test_cal). `lambda_df` has a
        single row (recency_lambda=0.0) when use_reweighting=False, since
        there is no lambda grid to search in that case.
        """
        X_train, X_valid, X_test = X[train_idx], X[valid_idx], X[test_idx]
        y_train, y_valid, y_test = y[train_idx], y[valid_idx], y[test_idx]
        t_train = time_rank[train_idx]

        recency_lambdas = self.recency_lambdas if use_reweighting else (0.0,)

        lambda_candidates = []
        stored = {}

        for rec_lam in recency_lambdas:
            if use_reweighting:
                w_train = combined_sample_weights(
                    y_train, t_train, recency_lambda=rec_lam,
                    beta=self.class_weight_beta, num_classes=self.num_classes,
                )
            else:
                w_train = np.ones(len(y_train), dtype=float)

            if model_kind == "da_minilm_mlp":
                valid_proba, test_proba, raw_valid_f1 = self.model.train_da_minilm_mlp(
                    X_train, y_train, w_train, X_valid, y_valid, X_test, seed=seed,
                )
                default_model_name = "DA-MiniLM-RWAC" if use_reweighting else "Uniform-MLP"
            elif model_kind == "minilm_logreg":
                valid_proba, test_proba = self.model.train_logreg_baseline(
                    X_train, y_train, w_train, X_valid, y_valid, X_test, seed=seed,
                )
                raw_valid_f1 = f1_score(y_valid, valid_proba.argmax(axis=1), average="macro", zero_division=0)
                default_model_name = "MiniLM-LogReg-RW" if use_reweighting else "Uniform-LR"
            else:
                raise ValueError(f"Unknown model_kind: {model_kind}")
            model_name = model_name_override or default_model_name

            if use_calibration:
                if calibration_mode == "joint_macro_f1":
                    calib = select_best_calibration(
                        valid_proba, y_valid, y_train, objective="macro_f1",
                        tau_grid=self.tau_grid, temperature_grid=self.temperature_grid, num_classes=self.num_classes,
                    )
                elif calibration_mode == "two_stage":
                    calib_raw = select_best_calibration_two_stage(
                        valid_proba, y_valid, y_train,
                        tau_grid=self.tau_grid, temperature_grid=self.temperature_grid, num_classes=self.num_classes,
                    )
                    calib = {"tau": calib_raw["tau"], "temperature": calib_raw["temperature"]}
                else:
                    raise ValueError(f"Unknown calibration_mode: {calibration_mode}")
            else:
                calib = {"tau": 0.0, "temperature": 1.0}

            valid_cal = apply_bias_and_temperature(
                valid_proba,
                bias=logit_adjustment_bias(y_train, tau=calib["tau"], num_classes=self.num_classes),
                temperature=calib["temperature"],
            )
            valid_macro_f1 = f1_score(y_valid, valid_cal.argmax(axis=1), average="macro", zero_division=0)

            lambda_candidates.append({
                "recency_lambda": rec_lam, "valid_macro_f1": valid_macro_f1,
                "raw_valid_macro_f1": raw_valid_f1, "tau": calib["tau"], "temperature": calib["temperature"],
            })
            stored[rec_lam] = (valid_proba, test_proba, calib)

        lambda_df = pd.DataFrame(lambda_candidates).sort_values("valid_macro_f1", ascending=False)
        best_lambda = float(lambda_df.iloc[0]["recency_lambda"])
        valid_proba, test_proba, calib = stored[best_lambda]

        bias = logit_adjustment_bias(y_train, tau=calib["tau"], num_classes=self.num_classes)
        test_cal = apply_bias_and_temperature(test_proba, bias=bias, temperature=calib["temperature"])
        pred_test = test_cal.argmax(axis=1)

        row = evaluate_predictions(
            y_test, pred_test, proba=test_cal, model_name=model_name, split_mode=split_mode,
            text_mode=text_mode, seed=seed, recency_lambda=best_lambda, tau=calib["tau"],
            temperature=calib["temperature"], num_classes=self.num_classes,
            extra={
                "n_train": len(train_idx), "n_valid": len(valid_idx), "n_test": len(test_idx),
                "selected_valid_macro_f1": float(lambda_df.iloc[0]["valid_macro_f1"]),
                "use_reweighting": use_reweighting, "use_calibration": use_calibration,
                "calibration_mode": calibration_mode if use_calibration else None,
            },
        )
        return row, lambda_df, pred_test, test_cal

    # ---------------- Component ablation ----------------

    def run_component_ablation(
        self, X, y, time_rank, train_idx, valid_idx, test_idx,
        text_mode: str, split_mode: str, seed: int, calibration_mode: str = "two_stage",
    ) -> pd.DataFrame:
        """The four ablation cells isolating each design choice's
        contribution, plus the two full models for reference:

            Uniform-LR         : LogReg, no reweighting, no calibration
            LogReg + RW         : LogReg, reweighting, no calibration
            MiniLM-LogReg-RW    : LogReg, reweighting, calibration (full)
            Uniform-MLP         : MLP,    no reweighting, no calibration
            MLP + RW            : MLP,    reweighting, no calibration
            DA-MiniLM-RWAC      : MLP,    reweighting, calibration (full)

        Reading down each architecture's three rows isolates reweighting's
        contribution (row 2 minus row 1) and calibration's contribution
        (row 3 minus row 2), holding architecture fixed. Reading across
        LogReg vs MLP at matched reweighting/calibration settings isolates
        the architecture's own contribution.
        """
        cells = [
            ("minilm_logreg", False, False, "Uniform-LR"),
            ("minilm_logreg", True, False, "LogReg + RW (no calibration)"),
            ("minilm_logreg", True, True, "MiniLM-LogReg-RW"),
            ("da_minilm_mlp", False, False, "Uniform-MLP"),
            ("da_minilm_mlp", True, False, "MLP + RW (no calibration)"),
            ("da_minilm_mlp", True, True, "DA-MiniLM-RWAC"),
        ]
        rows = []
        for model_kind, use_rw, use_calib, label in cells:
            print(f"running ablation cell: {label}")
            row, _, _, _ = self.run_single_experiment(
                X, y, time_rank, train_idx, valid_idx, test_idx,
                text_mode=text_mode, split_mode=split_mode, seed=seed, model_kind=model_kind,
                use_reweighting=use_rw, use_calibration=use_calib,
                calibration_mode=calibration_mode, model_name_override=label,
            )
            rows.append(row)
            print(f"  macro_f1={row['macro_f1']:.4f}")
        return pd.DataFrame(rows)

    # ---------------- Full battery ----------------

    def run_full_battery(
        self, df: pd.DataFrame, embeddings: dict, seeds: Sequence[int],
        text_modes: Sequence[str], split_modes: Sequence[str],
        model_kinds: Sequence[str] = ("minilm_logreg", "da_minilm_mlp"),
        calibration_mode: str = "two_stage",
    ):
        """`embeddings` must be a dict {text_mode: np.ndarray} (from
        model.encode_texts_minilm()). Splitting is delegated to
        self.splitter -- NOT reimplemented here. `df` must have a reset
        integer index (positions == train_idx/valid_idx/test_idx from the
        split functions), a `label_id` column, and a `time_rank` column.

        BUG FIX: this method previously did not pass calibration_mode to
        run_single_experiment() at all, so every call silently used
        run_single_experiment()'s own default of "joint_macro_f1" -- the
        calibration scheme where temperature has no effect on macro-F1 and
        is therefore never meaningfully selected (see
        select_best_calibration()'s docstring). run_component_ablation()
        already defaulted to the fixed "two_stage" scheme; run_full_battery
        (the method that actually produces your main/headline results) did
        not. Now it does, by default."""
        y_all = df["label_id"].values.astype(int)
        time_rank_all = df["time_rank"].values.astype(float)

        all_rows, all_lambda_rows = [], []
        prediction_store = {}

        for text_mode in text_modes:
            X = embeddings[text_mode]
            for split_mode in split_modes:
                for seed in seeds:
                    train_idx, valid_idx, test_idx = self.splitter.get_split(df, split_mode, seed=seed)

                    for model_kind in model_kinds:
                        print(f"running [{model_kind}] text={text_mode} split={split_mode} seed={seed}")
                        row, lambda_df, pred, proba = self.run_single_experiment(
                            X, y_all, time_rank_all, train_idx, valid_idx, test_idx,
                            text_mode=text_mode, split_mode=split_mode, seed=seed, model_kind=model_kind,
                            calibration_mode=calibration_mode,
                        )
                        all_rows.append(row)
                        for _, r in lambda_df.iterrows():
                            all_lambda_rows.append({
                                "model": row["model"], "text_mode": text_mode, "split_mode": split_mode,
                                "seed": seed, **r.to_dict(),
                            })
                        prediction_store[(row["model"], text_mode, split_mode, seed)] = {
                            "y_test": y_all[test_idx], "pred": pred, "proba": proba,
                        }
                        print(f"  macro_f1={row['macro_f1']:.4f}, ece={row['ece']:.4f}, lambda={row['recency_lambda']}")

        main_results_df = pd.DataFrame(all_rows)
        lambda_selection_df = pd.DataFrame(all_lambda_rows)

        if self.output_dir is not None:
            main_results_df.to_csv(f"{self.output_dir}/main_da_minilm_results.csv", index=False)
            lambda_selection_df.to_csv(f"{self.output_dir}/lambda_selection_validation_results.csv", index=False)

        return main_results_df, lambda_selection_df, prediction_store

    # ---------------- Summaries and comparisons ----------------

    def summarize_results(self, results_df: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
        metrics = ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy", "ece", "brier", "nll"]
        agg = {}
        for m in metrics:
            if m in results_df.columns:
                agg[f"{m}_mean"] = (m, "mean")
                agg[f"{m}_std"] = (m, "std")
                agg[f"{m}_min"] = (m, "min")
                agg[f"{m}_max"] = (m, "max")
        return results_df.groupby(group_cols).agg(**agg, n_runs=("macro_f1", "count")).reset_index()

    def run_paired_bootstrap(
        self, prediction_store: dict, text_modes: Sequence[str], split_modes: Sequence[str],
        seeds: Sequence[int], model_a: str = "DA-MiniLM-RWAC", model_b: str = "MiniLM-LogReg-RW",
        n_boot: int = 2000,
    ) -> pd.DataFrame:
        rows = []
        for text_mode in text_modes:
            for split_mode in split_modes:
                for seed in seeds:
                    key_a = (model_a, text_mode, split_mode, seed)
                    key_b = (model_b, text_mode, split_mode, seed)
                    if key_a not in prediction_store or key_b not in prediction_store:
                        continue
                    y_test = prediction_store[key_a]["y_test"]
                    boot = paired_bootstrap_macro_f1_diff(
                        y_test, prediction_store[key_a]["pred"], prediction_store[key_b]["pred"],
                        n_boot=n_boot, seed=seed,
                    )
                    rows.append({
                        "comparison": f"{model_a} minus {model_b}", "text_mode": text_mode,
                        "split_mode": split_mode, "seed": seed, **boot,
                    })
        bootstrap_df = pd.DataFrame(rows)
        if self.output_dir is not None:
            bootstrap_df.to_csv(f"{self.output_dir}/paired_bootstrap_results.csv", index=False)
        return bootstrap_df

    def run_rolling_window_evaluation(
        self, df: pd.DataFrame, embeddings: dict, text_mode: str, seeds: Sequence[int],
        model_kind: str = "da_minilm_mlp", rolling_windows_kwargs: Optional[dict] = None,
    ) -> pd.DataFrame:
        """Rolling windows are built by self.splitter.make_rolling_windows()
        -- not reimplemented here."""
        rolling_windows_kwargs = rolling_windows_kwargs or {}
        windows = self.splitter.make_rolling_windows(df, **rolling_windows_kwargs)

        y_all = df["label_id"].values.astype(int)
        time_rank_all = df["time_rank"].values.astype(float)
        X = embeddings[text_mode]

        rows = []
        for w in windows:
            for seed in seeds:
                print(f"running [rolling] window={w['window_id']} seed={seed}")
                row, lambda_df, pred, proba = self.run_single_experiment(
                    X, y_all, time_rank_all, w["train_idx"], w["valid_idx"], w["test_idx"],
                    text_mode=text_mode, split_mode="rolling_window", seed=seed, model_kind=model_kind,
                )
                row.update({
                    "window_id": w["window_id"], "train_end_frac": w["train_end_frac"],
                    "valid_end_frac": w["valid_end_frac"], "test_end_frac": w["test_end_frac"],
                })
                rows.append(row)
                print(f"  macro_f1={row['macro_f1']:.4f}, ece={row['ece']:.4f}, lambda={row['recency_lambda']}")

        rolling_results_df = pd.DataFrame(rows)
        if self.output_dir is not None:
            rolling_results_df.to_csv(f"{self.output_dir}/rolling_window_results.csv", index=False)
        return rolling_results_df


if __name__ == "__main__":
    print("da_minilm_experiment_runner.py loaded. Class: DAMiniLMExperimentRunner")

da_minilm_experiment_runner.py loaded. Class: DAMiniLMExperimentRunner


In [11]:
"""
shuffle_control.py

Shuffle-control (label permutation) validation, addressing this project's
tracked "remaining required experiment": establish a null distribution
for test macro-F1 by training on randomly permuted TRAIN labels (keeping
TEST labels real), repeated 200-500 times, and compare the real model's
test macro-F1 against that null distribution. If the real result isn't
comfortably above the null distribution's upper tail, the model isn't
demonstrably learning anything from the true label structure.

Deliberately model-agnostic: takes a `train_predict_fn` callable rather
than hard-coding DAMiniLMModel, so the same validator works for the
frozen-embedding LogReg baseline (fast, recommended for 200-500 shuffles),
the MLP (much slower -- consider fewer shuffles or a subsample), or any
future model. `train_predict_fn` must accept
(X_train, y_train, X_valid, y_valid, X_test, seed) and return
(valid_proba, test_proba).

Usage
-----
    from shuffle_control import ShuffleControlValidator
    from da_minilm_model import DAMiniLMModel

    model = DAMiniLMModel(output_dir="./da_minilm_results")

    def train_predict_fn(X_train, y_train, X_valid, y_valid, X_test, seed):
        w_train = np.ones(len(y_train))  # shuffle control doesn't reweight
        return model.train_logreg_baseline(X_train, y_train, w_train, X_valid, y_valid, X_test, seed=seed)

    validator = ShuffleControlValidator(train_predict_fn, n_shuffles=300, seed=42)
    result = validator.run(X_train, y_train, X_valid, y_valid, X_test, y_test)
    print(result["summary"])
"""

from typing import Callable, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


class ShuffleControlValidator:

    def __init__(self, train_predict_fn: Callable, n_shuffles: int = 300, seed: int = 42):
        """
        train_predict_fn(X_train, y_train, X_valid, y_valid, X_test, seed)
            -> (valid_proba, test_proba)
        Any function with this signature works -- e.g. a lambda wrapping
        DAMiniLMModel.train_logreg_baseline() or .train_da_minilm_mlp()
        with fixed/uniform sample weights, since shuffle control tests
        whether the LABEL structure matters, not whether reweighting does.
        """
        self.train_predict_fn = train_predict_fn
        self.n_shuffles = n_shuffles
        self.seed = seed

    def run(
        self, X_train, y_train, X_valid, y_valid, X_test, y_test,
        real_test_macro_f1: Optional[float] = None,
    ) -> dict:
        """Trains `n_shuffles` times on a randomly permuted copy of
        y_train (test labels are never touched), collecting test macro-F1
        each time to build the null distribution. If `real_test_macro_f1`
        isn't supplied, it's computed here by running train_predict_fn
        once more on the REAL (unshuffled) labels, so the comparison is
        always apples-to-apples on the same X_test/y_test.
        """
        rng = np.random.RandomState(self.seed)

        if real_test_macro_f1 is None:
            _, real_test_proba = self.train_predict_fn(X_train, y_train, X_valid, y_valid, X_test, self.seed)
            real_test_macro_f1 = f1_score(y_test, real_test_proba.argmax(axis=1), average="macro", zero_division=0)

        null_scores = np.empty(self.n_shuffles)
        for i in range(self.n_shuffles):
            shuffled_y_train = rng.permutation(y_train)
            shuffle_seed = int(rng.randint(0, 2**31 - 1))
            _, test_proba = self.train_predict_fn(
                X_train, shuffled_y_train, X_valid, y_valid, X_test, shuffle_seed,
            )
            null_scores[i] = f1_score(y_test, test_proba.argmax(axis=1), average="macro", zero_division=0)
            if (i + 1) % max(1, self.n_shuffles // 10) == 0:
                print(f"  shuffle {i + 1}/{self.n_shuffles}: macro_f1={null_scores[i]:.4f}")

        p_value = float((np.sum(null_scores >= real_test_macro_f1) + 1) / (self.n_shuffles + 1))

        summary = {
            "n_shuffles": self.n_shuffles,
            "real_test_macro_f1": float(real_test_macro_f1),
            "null_mean": float(null_scores.mean()),
            "null_std": float(null_scores.std()),
            "null_p95": float(np.percentile(null_scores, 95)),
            "null_p99": float(np.percentile(null_scores, 99)),
            "null_max": float(null_scores.max()),
            "p_value": p_value,
            "real_exceeds_null_p95": bool(real_test_macro_f1 > np.percentile(null_scores, 95)),
            "real_exceeds_null_max": bool(real_test_macro_f1 > null_scores.max()),
        }

        return {"summary": summary, "null_scores": null_scores}

    @staticmethod
    def print_summary(result: dict) -> None:
        s = result["summary"]
        print(f"Real test macro-F1:        {s['real_test_macro_f1']:.4f}")
        print(f"Null distribution (n={s['n_shuffles']}): "
              f"mean={s['null_mean']:.4f}, std={s['null_std']:.4f}, "
              f"95th pct={s['null_p95']:.4f}, 99th pct={s['null_p99']:.4f}, max={s['null_max']:.4f}")
        print(f"p-value (fraction of shuffles >= real): {s['p_value']:.4f}")
        print(f"Real result exceeds null 95th percentile: {s['real_exceeds_null_p95']}")
        print(f"Real result exceeds null maximum:          {s['real_exceeds_null_max']}")


if __name__ == "__main__":
    print("shuffle_control.py loaded. Class: ShuffleControlValidator")

shuffle_control.py loaded. Class: ShuffleControlValidator


In [12]:
"""
baseline_models.py

Unified baseline-model class covering LIAR, LIAR2, and the LIAR+LIAR2
merge -- every model x dataset combination in
baseline_model_comparison_liar_liar2.xlsx, INCLUDING B17/B18.

B17/B18 were initially assumed to be incomplete runs (split_type="To be
run" in the xlsx, B18's oddly small n_test=203), but checking
cross-dataset-tf-idf-logistic-regression.ipynb directly showed they are
actually CROSS-DATASET GENERALIZATION experiments: B17 trains on LIAR and
tests on LIAR2; B18 trains on LIAR2 and tests on LIAR. Both remove target-
test rows that overlap the source train/valid before evaluating -- which
is exactly why B18 has so few test rows: LIAR2 extends LIAR, so most of
LIAR's test statements already appear in LIAR2's train/valid and get
correctly stripped out as leakage. See run_cross_dataset_generalization().

Covered (model x dataset):
    TF-IDF + Logistic Regression         -> LIAR (B01), LIAR2 (B02),
                                             raw merge (B03), clean merge (B04)
    TF-IDF + Linear SVM                  -> LIAR (B05), LIAR2 (B06),
                                             clean merge (B07)
    TF-IDF + Logistic Regression + meta  -> LIAR (B08), LIAR2 (B09),
                                             clean merge (B10)
    DistilBERT                           -> LIAR (B11), LIAR2 (B12),
                                             clean merge (B13)
    MiniLM                                -> LIAR (B14), LIAR2 (B15),
                                             clean merge (B16)
    Chronological (LIAR2 only, 70/15/15) -> LogReg (C01), Linear SVM (C02)

ALL data loading/cleaning goes through LiarPreprocessor and
Liar2Preprocessor -- this file contains no re-implementation of cleaning
logic. It only builds the LIAR+LIAR2 merge (schema harmonization +
cross-dataset dedup) on top of what those two produce, then fits/evaluates
models.

Design decisions worth knowing
-------------------------------
1. "Statement + metadata" uses only each dataset's leakage-safe metadata
   columns (LiarPreprocessor.SAFE_METADATA_COLUMNS /
   Liar2Preprocessor.SAFE_METADATA_COLUMNS), never the credibility-history
   counts. For the merge, the metadata columns are further restricted to
   the INTERSECTION of both datasets' safe columns (subject, speaker,
   state_info, context) -- LIAR's job_title/party_affiliation and LIAR2's
   speaker_description don't exist on both sides, so they're dropped for
   the merged condition specifically, not globally.

2. build_liar_liar2_merge() reproduces the "clean merge" dedup behavior
   from tf-idf-logistic-regression-lin-svm-cron-split.ipynb: normalize
   statement text, prefer the dated (LIAR2) version of a cross-dataset
   duplicate over the undated (LIAR) version, keep the earliest by date
   among LIAR2-only duplicates. Verified against your actual
   liar_liar2_clean_merged_tfidf.csv: this reproduces the exact same
   total (23,886 rows) and the exact same LIAR2/LIAR composition
   (13,127 / 10,759) and split sizes (21,053 / 1,539 / 1,294) -- see the
   test block at the bottom of this file's companion test run.

3. The RAW merge (no dedup) is included because it's B03 in the xlsx,
   but B03's own leakage_status says "Leakage detected" and its
   reporting_status says "Do not report as final" -- run_raw_merge_baseline()
   exists so you can regenerate that leakage DEMONSTRATION, not because
   it's a result to cite.

4. Chronological splitting is ONLY valid for LIAR2 -- LIAR has no date
   column. run_tfidf_logreg(..., split_mode="chronological") on a
   LIAR-only or merged DataFrame raises a clear error rather than silently
   doing something else.

5. DistilBERT and MiniLM require Hugging Face Hub access to download
   pretrained weights, which this sandbox's network cannot reach --
   written to match baseline_DistilBERT_Text-Only.ipynb's structure and
   this project's DA-MiniLM Uniform-LR convention, but NOT executed here.
   Test them in your own GPU/internet-enabled environment before trusting
   their output. Everything else (the five TF-IDF-based experiment
   families) is tested end-to-end against your real uploaded files.

Usage
-----
    from liar_preprocessor import LiarPreprocessor
    from liar2_preprocessor import Liar2Preprocessor
    from baseline_models import BaselineModels

    liar_prep = LiarPreprocessor()
    liar_df = liar_prep.run_pipeline(liar_train, liar_valid, liar_test)

    liar2_prep = Liar2Preprocessor()
    liar2_df = liar2_prep.run_pipeline(liar2_train, liar2_valid, liar2_test)

    baselines = BaselineModels(liar_preprocessor=liar_prep, liar2_preprocessor=liar2_prep)

    merged_clean_df = baselines.build_liar_liar2_merge(liar_df, liar2_df, deduplicate=True)

    results_df = baselines.run_all_experiments(
        liar_df=liar_df, liar2_df=liar2_df,
        include_transformer_baselines=False,
    )
"""

import re
from typing import Optional, Sequence

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


class BaselineModels:

    # ----------------------------------------------------------------
    # Model configs. NOTE: solver="liblinear" from the original notebooks
    # does not run on current scikit-learn (>=1.5; this sandbox has 1.8.0)
    # for 6-class problems -- liblinear's one-vs-rest multiclass fallback
    # was removed outright, not just deprecated. Swapped to "lbfgs"
    # throughout, confirmed by running it here.
    # ----------------------------------------------------------------

    TFIDF_LOGREG_VECTORIZER = dict(
        lowercase=True, stop_words="english", ngram_range=(1, 2),
        max_features=30000, min_df=2, max_df=0.90, sublinear_tf=True,
    )
    TFIDF_LOGREG_CLASSIFIER = dict(
        C=1.0, max_iter=2000, class_weight="balanced", solver="lbfgs",
    )

    TFIDF_SVM_VECTORIZER = dict(
        lowercase=True, stop_words=None, ngram_range=(1, 2),
        max_features=30000, min_df=2, max_df=0.90, sublinear_tf=True,
    )
    TFIDF_SVM_CLASSIFIER = dict(
        C=1.0, class_weight="balanced", max_iter=5000, random_state=42,
    )

    # Chronological battery (C01/C02) preset, matching
    # tf-idf-logistic-regression-lin-svm-cron-split.ipynb exactly (except
    # the same liblinear -> lbfgs swap).
    TFIDF_LOGREG_VECTORIZER_CHRONO = dict(
        lowercase=True, strip_accents="unicode", stop_words="english",
        ngram_range=(1, 2), max_features=50000, min_df=2, max_df=0.95,
        sublinear_tf=True,
    )
    TFIDF_LOGREG_CLASSIFIER_CHRONO = dict(
        solver="lbfgs", C=1.0, max_iter=3000, random_state=42,
    )
    TFIDF_SVM_VECTORIZER_CHRONO = TFIDF_LOGREG_VECTORIZER_CHRONO
    TFIDF_SVM_CLASSIFIER_CHRONO = dict(C=1.0, random_state=42)

    TFIDF_METADATA_VECTORIZER = dict(
        lowercase=True, stop_words="english", ngram_range=(1, 2),
        min_df=2, max_df=0.95, max_features=50000, sublinear_tf=True,
    )
    LOGREG_METADATA_CLASSIFIER = dict(
        solver="saga", C=1.0, class_weight="balanced",
        max_iter=5000, random_state=42,
    )

    # Metadata columns common to BOTH LIAR and LIAR2 -- used for the merged
    # ("statement + metadata") condition specifically. LIAR's job_title /
    # party_affiliation and LIAR2's speaker_description are dropped only
    # in this merged context, not from the single-dataset runs.
    MERGE_COMMON_METADATA_COLUMNS = ["subject", "speaker", "state_info", "context"]

    def __init__(self, liar_preprocessor=None, liar2_preprocessor=None):
        """Pass whichever preprocessor(s) you have. `liar2_preprocessor`
        is required for chronological_split (LIAR2-only feature)."""
        self.liar_p = liar_preprocessor
        self.liar2_p = liar2_preprocessor

    # ================================================================
    # LIAR + LIAR2 merge builder
    # ================================================================

    @staticmethod
    def _normalize_for_overlap(x: str) -> str:
        """Matches baseline_model_TF-IDF_Linear_SVM.ipynb's normalize_text()
        EXACTLY, including operation order: lowercase, collapse whitespace,
        THEN strip non-word characters using \\w (Unicode-aware in Python 3
        by default -- this matters: an ASCII-only [^a-z0-9\\s] regex strips
        accented characters entirely instead of preserving them, which
        changes which statements collide as "duplicates." Confirmed this
        was the exact source of a merge-count mismatch (23,551 vs the
        correct 23,886) during testing -- do not simplify this to an
        ASCII-only character class."""
        x = str(x).lower()
        x = re.sub(r"\s+", " ", x)
        x = re.sub(r"[^\w\s]", "", x)
        return x.strip()

    def build_liar_liar2_merge(
        self, liar_df: pd.DataFrame, liar2_df: pd.DataFrame, deduplicate: bool = True,
    ) -> pd.DataFrame:
        """Harmonize LIAR and LIAR2 to a common schema and concatenate,
        matching the EXACT dedup algorithm from
        baseline_model_TF-IDF_Linear_SVM.ipynb's "Fresh Clean LIAR + LIAR2
        Combined" section (verified byte-for-byte against
        liar_liar2_clean_merged_tfidf.csv: 23,886 total rows, LIAR2=13,127
        / LIAR=10,759, split 21,053 / 1,539 / 1,294 -- exact match).

        The algorithm, in order (this exact order matters):
        1. Concatenate LIAR FIRST, then LIAR2 (not the other way around --
           this determines which row "wins" a duplicate: LIAR wins,
           because drop_duplicates(keep="first") sees LIAR rows first).
        2. Split into train/valid/test using official_split.
        3. Within EACH split separately, drop duplicate normalized
           statements, keeping the first occurrence (i.e. preferring LIAR
           over LIAR2 when both contain the same claim in the same split).
        4. THEN remove cross-split leakage: valid rows whose normalized
           statement appears in the (deduped) train are dropped; test rows
           whose normalized statement appears in the deduped train OR the
           now-cleaned valid are dropped.
        This is NOT a simple "dedupe the whole merged frame" pass -- doing
        it in the wrong order (e.g. date-preference, or deduping before
        splitting) gives a different, wrong cohort. An earlier version of
        this function used a date-preference heuristic and got 23,551
        rows with the composition nearly inverted (LIAR2=22,923,
        LIAR=628) -- confirmed wrong against the real file, replaced by
        the algorithm above.
        """
        common_cols = ["statement", "label", "label_id", "official_split"]

        liar_part = liar_df[[c for c in common_cols if c in liar_df.columns]].copy()
        for col in self.MERGE_COMMON_METADATA_COLUMNS:
            liar_part[col] = liar_df[col] if col in liar_df.columns else np.nan
        liar_part["dataset_source"] = "LIAR"

        liar2_part = liar2_df[[c for c in common_cols if c in liar2_df.columns]].copy()
        for col in self.MERGE_COMMON_METADATA_COLUMNS:
            liar2_part[col] = liar2_df[col] if col in liar2_df.columns else np.nan
        liar2_part["dataset_source"] = "LIAR2"

        # LIAR FIRST -- order determines which row wins a same-split duplicate.
        merged = pd.concat([liar_part, liar2_part], ignore_index=True)
        n_raw = len(merged)
        print(f"Raw merge: {n_raw} rows (LIAR={len(liar_part)}, LIAR2={len(liar2_part)})")

        if not deduplicate:
            print("Composition (raw, no dedup):")
            print(merged["dataset_source"].value_counts())
            return merged

        merged["_norm"] = merged["statement"].apply(self._normalize_for_overlap)

        train_df = merged[merged["official_split"] == "train"].drop_duplicates(subset="_norm", keep="first").copy()
        valid_df = merged[merged["official_split"] == "valid"].drop_duplicates(subset="_norm", keep="first").copy()
        test_df = merged[merged["official_split"] == "test"].drop_duplicates(subset="_norm", keep="first").copy()

        train_statements = set(train_df["_norm"])
        valid_df = valid_df[~valid_df["_norm"].isin(train_statements)].copy()
        valid_statements_clean = set(valid_df["_norm"])
        test_df = test_df[~test_df["_norm"].isin(train_statements)].copy()
        test_df = test_df[~test_df["_norm"].isin(valid_statements_clean)].copy()

        final = pd.concat([train_df, valid_df, test_df], ignore_index=True).drop(columns=["_norm"])

        print(f"Clean merge: {n_raw} -> {len(final)} (removed {n_raw - len(final)})")
        print("Clean merge composition:")
        print(final["dataset_source"].value_counts())
        print("Clean merge split composition:")
        print(final["official_split"].value_counts().reindex(["train", "valid", "test"]))

        return final

    # ================================================================
    # Cross-dataset generalization (B17: train LIAR -> test LIAR2;
    # B18: train LIAR2 -> test LIAR), matching
    # cross-dataset-tf-idf-logistic-regression.ipynb
    # ================================================================

    def run_cross_dataset_generalization(
        self, source_df: pd.DataFrame, source_name: str,
        target_df: pd.DataFrame, target_name: str,
        remove_target_overlap: bool = True, text_col: str = "statement",
    ) -> dict:
        """Train on `source_df`'s official train+valid, test on
        `target_df`'s official test. Matches
        cross-dataset-tf-idf-logistic-regression.ipynb's B17/B18 design
        exactly: source train/valid are used as-is; if
        remove_target_overlap=True (the notebook's default), target test
        rows whose normalized statement appears in source train OR source
        valid are dropped before evaluation -- this is a LEAKAGE CHECK ON
        THE TARGET TEST SET, not a dedup of the source data, and it is
        the reason B18 (train LIAR2 -> test LIAR) has a much smaller
        n_test than B17 (train LIAR -> test LIAR2): LIAR2 extends LIAR, so
        most of LIAR's test statements already appear somewhere in LIAR2's
        train/valid.

        `source_name`/`target_name` are just labels for the result row
        (e.g. "LIAR", "LIAR2").
        """
        source_train = source_df[source_df["official_split"] == "train"].copy()
        source_valid = source_df[source_df["official_split"] == "valid"].copy()
        target_test = target_df[target_df["official_split"] == "test"].copy()

        for part in (source_train, source_valid, target_test):
            part["_norm"] = part[text_col].apply(self._normalize_for_overlap)

        overlap_before = {
            "train_valid_overlap": len(set(source_train["_norm"]) & set(source_valid["_norm"])),
            "train_test_overlap": len(set(source_train["_norm"]) & set(target_test["_norm"])),
            "valid_test_overlap": len(set(source_valid["_norm"]) & set(target_test["_norm"])),
        }

        removed_target_rows = 0
        if remove_target_overlap:
            source_seen = set(source_train["_norm"]) | set(source_valid["_norm"])
            before = len(target_test)
            target_test = target_test[~target_test["_norm"].isin(source_seen)].copy()
            removed_target_rows = before - len(target_test)
            print(f"Cross-dataset ({source_name} -> {target_name}): removed {removed_target_rows} "
                  f"target-test rows overlapping source train/valid ({before} -> {len(target_test)}).")

        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(**self.TFIDF_LOGREG_VECTORIZER_CHRONO)),  # matches the notebook's config
            ("classifier", LogisticRegression(solver="lbfgs", C=1.0, max_iter=3000, random_state=42)),
        ])
        pipeline.fit(source_train[text_col], source_train["label"])

        valid_metrics = self._evaluate(source_valid["label"], pipeline.predict(source_valid[text_col]))
        test_metrics = self._evaluate(target_test["label"], pipeline.predict(target_test[text_col]))
        overlap_after = self.compute_overlap(source_train, source_valid, target_test)

        return self._assemble_result_row(
            "TF-IDF + Logistic Regression", f"Train {source_name} -> Test {target_name}",
            "Cross-dataset generalization", "statement only",
            source_train, source_valid, target_test, valid_metrics, test_metrics, overlap_after,
            notes=f"Target-test leakage removal: {'ON' if remove_target_overlap else 'OFF'}. "
                  f"Removed {removed_target_rows} target-test rows overlapping source train/valid. "
                  f"Overlap before cleaning: {overlap_before}.",
        )

    # ================================================================
    # Split retrieval
    # ================================================================

    def get_split(self, df: pd.DataFrame, split_mode: str = "official", seed: int = 42):
        """
        split_mode:
            "official"           -> uses df["official_split"] as-is
            "random_stratified"  -> generic 70/15/15 stratified re-split
                                     (works for LIAR, LIAR2, or the merge)
            "chronological"      -> ONLY valid for a pure-LIAR2 DataFrame
                                     with a `date_parsed` column; delegates
                                     to self.liar2_p.chronological_split().
                                     Raises if liar2_preprocessor wasn't
                                     provided, or if df has no date_parsed
                                     column (i.e. it's LIAR-only or merged).
        """
        if split_mode == "official":
            if "official_split" not in df.columns:
                raise ValueError("df has no 'official_split' column.")
            train_df = df[df["official_split"] == "train"].copy()
            valid_df = df[df["official_split"] == "valid"].copy()
            test_df = df[df["official_split"] == "test"].copy()
            return train_df, valid_df, test_df, "Official train/valid/test"

        if split_mode == "random_stratified":
            all_idx = np.arange(len(df))
            y = df["label_id"].values
            train_idx, temp_idx = train_test_split(all_idx, train_size=0.70, random_state=seed, stratify=y)
            y_temp = y[temp_idx]
            valid_idx, test_idx = train_test_split(temp_idx, train_size=0.5, random_state=seed, stratify=y_temp)
            train_idx, valid_idx, test_idx = np.sort(train_idx), np.sort(valid_idx), np.sort(test_idx)
            return (df.iloc[train_idx].copy(), df.iloc[valid_idx].copy(), df.iloc[test_idx].copy(),
                    "Random stratified 70/15/15 split")

        if split_mode == "chronological":
            if "date_parsed" not in df.columns:
                raise ValueError(
                    "chronological split requires a 'date_parsed' column -- only pure LIAR2 "
                    "DataFrames have one. LIAR has no date column, and the merge doesn't have "
                    "a coherent one either (LIAR rows have no date)."
                )
            if self.liar2_p is None:
                raise ValueError("chronological split requires liar2_preprocessor to be set on this instance.")
            train_idx, valid_idx, test_idx = self.liar2_p.chronological_split(df)
            return (df.iloc[train_idx].copy(), df.iloc[valid_idx].copy(), df.iloc[test_idx].copy(),
                    "Chronological 70/15/15 split by date")

        raise ValueError(f"Unknown split_mode: {split_mode}")

    def compute_overlap(self, train_df, valid_df, test_df) -> dict:
        norm = self._normalize_for_overlap
        train_set = set(train_df["statement"].apply(norm))
        valid_set = set(valid_df["statement"].apply(norm))
        test_set = set(test_df["statement"].apply(norm))
        return {
            "train_valid_overlap": len(train_set & valid_set),
            "train_test_overlap": len(train_set & test_set),
            "valid_test_overlap": len(valid_set & test_set),
        }

    # ================================================================
    # Metrics
    # ================================================================

    def _evaluate(self, y_true, y_pred) -> dict:
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        }

    def _assemble_result_row(
        self, model_name, train_dataset_label, split_type_label, input_features,
        train_df, valid_df, test_df, valid_metrics, test_metrics, overlap, notes="",
    ) -> dict:
        return {
            "model": model_name,
            "train_dataset": train_dataset_label,
            "split_type": split_type_label,
            "input_features": input_features,
            "label_space": "6-class truthfulness",
            "n_train": len(train_df),
            "n_valid": len(valid_df),
            "n_test": len(test_df),
            "validation_accuracy": valid_metrics["accuracy"],
            "validation_macro_f1": valid_metrics["macro_f1"],
            "validation_weighted_f1": valid_metrics["weighted_f1"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_macro_precision": test_metrics["macro_precision"],
            "test_macro_recall": test_metrics["macro_recall"],
            "train_valid_overlap": overlap["train_valid_overlap"],
            "train_test_overlap": overlap["train_test_overlap"],
            "valid_test_overlap": overlap["valid_test_overlap"],
            "notes": notes,
        }

    # ================================================================
    # Classical baselines -- dataset-agnostic, fully tested
    # ================================================================

    def run_tfidf_logreg(
        self, df: pd.DataFrame, train_dataset_label: str, split_mode: str = "official",
        text_col: str = "statement", use_chrono_config: Optional[bool] = None,
    ) -> dict:
        """TF-IDF + Logistic Regression. Works on LIAR (B01), LIAR2 (B02),
        raw or clean merge (B03/B04), or LIAR2 chronological (C01) --
        pass whichever DataFrame and split_mode fits."""
        if use_chrono_config is None:
            use_chrono_config = split_mode == "chronological"
        vec_cfg = self.TFIDF_LOGREG_VECTORIZER_CHRONO if use_chrono_config else self.TFIDF_LOGREG_VECTORIZER
        clf_cfg = self.TFIDF_LOGREG_CLASSIFIER_CHRONO if use_chrono_config else self.TFIDF_LOGREG_CLASSIFIER

        train_df, valid_df, test_df, split_type_label = self.get_split(df, split_mode)

        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(**vec_cfg)),
            ("classifier", LogisticRegression(**clf_cfg)),
        ])
        pipeline.fit(train_df[text_col], train_df["label"])

        valid_metrics = self._evaluate(valid_df["label"], pipeline.predict(valid_df[text_col]))
        test_metrics = self._evaluate(test_df["label"], pipeline.predict(test_df[text_col]))
        overlap = self.compute_overlap(train_df, valid_df, test_df)

        return self._assemble_result_row(
            "TF-IDF + Logistic Regression", train_dataset_label, split_type_label, "statement only",
            train_df, valid_df, test_df, valid_metrics, test_metrics, overlap,
        )

    def run_tfidf_linear_svm(
        self, df: pd.DataFrame, train_dataset_label: str, split_mode: str = "official",
        text_col: str = "statement", use_chrono_config: Optional[bool] = None,
    ) -> dict:
        """TF-IDF + Linear SVM. Works on LIAR (B05), LIAR2 (B06), clean
        merge (B07), or LIAR2 chronological (C02)."""
        if use_chrono_config is None:
            use_chrono_config = split_mode == "chronological"
        vec_cfg = self.TFIDF_SVM_VECTORIZER_CHRONO if use_chrono_config else self.TFIDF_SVM_VECTORIZER
        clf_cfg = self.TFIDF_SVM_CLASSIFIER_CHRONO if use_chrono_config else self.TFIDF_SVM_CLASSIFIER

        train_df, valid_df, test_df, split_type_label = self.get_split(df, split_mode)

        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(**vec_cfg)),
            ("classifier", LinearSVC(**clf_cfg)),
        ])
        pipeline.fit(train_df[text_col], train_df["label"])

        valid_metrics = self._evaluate(valid_df["label"], pipeline.predict(valid_df[text_col]))
        test_metrics = self._evaluate(test_df["label"], pipeline.predict(test_df[text_col]))
        overlap = self.compute_overlap(train_df, valid_df, test_df)

        return self._assemble_result_row(
            "TF-IDF + Linear SVM", train_dataset_label, split_type_label, "statement only",
            train_df, valid_df, test_df, valid_metrics, test_metrics, overlap,
        )

    def _make_onehot_encoder(self):
        try:
            return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=True)

    def run_tfidf_logreg_metadata(
        self, df: pd.DataFrame, train_dataset_label: str, metadata_cols: Sequence[str],
        split_mode: str = "official", text_col: str = "statement",
    ) -> dict:
        """TF-IDF (text) + one-hot metadata + Logistic Regression. Works on
        LIAR (B08, pass LiarPreprocessor.SAFE_METADATA_COLUMNS), LIAR2
        (B09, pass Liar2Preprocessor.SAFE_METADATA_COLUMNS), or the merge
        (B10, pass BaselineModels.MERGE_COMMON_METADATA_COLUMNS).
        `metadata_cols` is passed explicitly rather than inferred, since it
        differs by dataset."""
        train_df, valid_df, test_df, split_type_label = self.get_split(df, split_mode)
        metadata_cols = [c for c in metadata_cols if c in df.columns]

        preprocessor = ColumnTransformer(
            transformers=[
                ("tfidf_text", TfidfVectorizer(**self.TFIDF_METADATA_VECTORIZER), text_col),
                ("categorical_metadata", Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", self._make_onehot_encoder()),
                ]), metadata_cols),
            ],
            remainder="drop",
            sparse_threshold=1.0,
        )
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(**self.LOGREG_METADATA_CLASSIFIER)),
        ])
        pipeline.fit(train_df, train_df["label"])

        valid_metrics = self._evaluate(valid_df["label"], pipeline.predict(valid_df))
        test_metrics = self._evaluate(test_df["label"], pipeline.predict(test_df))
        overlap = self.compute_overlap(train_df, valid_df, test_df)

        return self._assemble_result_row(
            "TF-IDF + Logistic Regression + metadata", train_dataset_label, split_type_label,
            "statement + metadata", train_df, valid_df, test_df, valid_metrics, test_metrics, overlap,
            notes=f"Metadata columns used (leakage-safe only): {metadata_cols}.",
        )

    # ================================================================
    # Transformer baselines -- WRITTEN BUT NOT RUNNABLE IN THIS SANDBOX
    # (no network access to huggingface.co). Test before use.
    # ================================================================

    def run_distilbert(
        self, df: pd.DataFrame, train_dataset_label: str, split_mode: str = "official",
        text_col: str = "statement", model_name: str = "distilbert-base-uncased",
        max_length: int = 128, batch_size: int = 16, epochs: int = 3,
        label_order: Sequence[str] = None, seed: int = 42,
    ) -> dict:
        """Fine-tuned DistilBERT, matching baseline_DistilBERT_Text-Only
        .ipynb. Works on LIAR (B11), LIAR2 (B12), clean merge (B13), or
        -- with split_mode="chronological" on a LIAR2 DataFrame -- the
        DistilBERT-chronological run this project's methodology tracks as
        a required (previously outstanding) experiment. NOT TESTED in this
        sandbox -- verify in your own GPU/internet environment first.
        `label_order` defaults to the canonical 6-class LIAR2/LIAR order
        if not given. `seed` controls torch's RNG for weight init and data
        shuffling -- run this multiple times with different seeds (see
        run_distilbert_multiseed() below) for the same multi-seed rigor
        used elsewhere in this project, rather than reporting a single run."""
        import torch
        from datasets import Dataset
        from transformers import (
            AutoTokenizer, AutoModelForSequenceClassification,
            TrainingArguments, Trainer, DataCollatorWithPadding,
        )

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        label_order = label_order or ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
        label_to_id = {l: i for i, l in enumerate(label_order)}

        train_df, valid_df, test_df, split_type_label = self.get_split(df, split_mode)
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        def to_hf_dataset(part_df):
            ds = Dataset.from_pandas(part_df[[text_col, "label"]].rename(columns={text_col: "text"}))
            ds = ds.map(lambda ex: {"label": label_to_id[ex["label"]]})
            ds = ds.map(lambda ex: tokenizer(ex["text"], truncation=True, max_length=max_length), batched=True)
            return ds

        train_ds, valid_ds, test_ds = to_hf_dataset(train_df), to_hf_dataset(valid_df), to_hf_dataset(test_df)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_order))
        collator = DataCollatorWithPadding(tokenizer=tokenizer)
        # transformers renamed `evaluation_strategy=` to `eval_strategy=` in
        # 4.46+, and separately renamed Trainer's `tokenizer=` to
        # `processing_class=` in 5.x. Both are handled defensively below
        # rather than pinned to one version, since this method can't be
        # tested against the Hugging Face Hub in this sandbox and the
        # Kaggle environment's exact transformers version is unknown.
        try:
            args = TrainingArguments(
                output_dir="./distilbert_baseline_out", num_train_epochs=epochs,
                per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
                eval_strategy="epoch", save_strategy="no", report_to=[], seed=seed,
            )
        except TypeError:
            args = TrainingArguments(
                output_dir="./distilbert_baseline_out", num_train_epochs=epochs,
                per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
                evaluation_strategy="epoch", save_strategy="no", report_to=[], seed=seed,
            )
        try:
            trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                               data_collator=collator, processing_class=tokenizer)
        except TypeError:
            trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                               data_collator=collator, tokenizer=tokenizer)
        trainer.train()

        id_to_label = {i: l for l, i in label_to_id.items()}
        valid_pred = [id_to_label[i] for i in np.argmax(trainer.predict(valid_ds).predictions, axis=1)]
        test_pred = [id_to_label[i] for i in np.argmax(trainer.predict(test_ds).predictions, axis=1)]

        valid_metrics = self._evaluate(valid_df["label"], valid_pred)
        test_metrics = self._evaluate(test_df["label"], test_pred)
        overlap = self.compute_overlap(train_df, valid_df, test_df)

        return self._assemble_result_row(
            "DistilBERT", train_dataset_label, split_type_label, "statement only",
            train_df, valid_df, test_df, valid_metrics, test_metrics, overlap,
            notes=f"epochs={epochs}, batch_size={batch_size}, max_length={max_length}, "
                  f"model={model_name}, seed={seed}.",
        )

    def run_distilbert_multiseed(
        self, df: pd.DataFrame, train_dataset_label: str, seeds, split_mode: str = "official",
        **kwargs,
    ) -> pd.DataFrame:
        """Runs run_distilbert() once per seed, returning a DataFrame with
        one row per seed plus mean/std summary rows -- matching the
        multi-seed rigor used for DA-MiniLM-RWAC/MiniLM-LogReg-RW
        elsewhere in this project, rather than reporting a single
        DistilBERT run. Use split_mode="chronological" on a LIAR2
        DataFrame for the DistilBERT-chronological run tracked as a
        required experiment."""
        rows = []
        for seed in seeds:
            print(f"running DistilBERT seed={seed} split_mode={split_mode}")
            row = self.run_distilbert(df, train_dataset_label, split_mode=split_mode, seed=seed, **kwargs)
            rows.append(row)
            print(f"  macro_f1={row['test_macro_f1']:.4f}")

        results_df = pd.DataFrame(rows)

        numeric_cols = [c for c in results_df.columns if results_df[c].dtype.kind in "fc"]
        summary = {"model": f"{results_df['model'].iloc[0]} (mean over {len(seeds)} seeds)",
                   "train_dataset": train_dataset_label, "split_type": results_df["split_type"].iloc[0]}
        for col in numeric_cols:
            summary[f"{col}_mean"] = results_df[col].mean()
            summary[f"{col}_std"] = results_df[col].std()

        print()
        print(f"DistilBERT {train_dataset_label} ({split_mode}), {len(seeds)} seeds: "
              f"test_macro_f1 = {summary['test_macro_f1_mean']:.4f} +/- {summary['test_macro_f1_std']:.4f}")

        return results_df, summary

    def run_minilm(
        self, df: pd.DataFrame, train_dataset_label: str, split_mode: str = "official",
        text_col: str = "statement", model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        batch_size: int = 256,
    ) -> dict:
        """Frozen MiniLM embeddings + Logistic Regression (Uniform-LR
        equivalent). Works on LIAR (B14), LIAR2 (B15), or clean merge
        (B16). NOT TESTED in this sandbox."""
        from sentence_transformers import SentenceTransformer

        train_df, valid_df, test_df, split_type_label = self.get_split(df, split_mode)

        embedder = SentenceTransformer(model_name)
        X_train = embedder.encode(train_df[text_col].tolist(), batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=False)
        X_valid = embedder.encode(valid_df[text_col].tolist(), batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=False)
        X_test = embedder.encode(test_df[text_col].tolist(), batch_size=batch_size,
                                  normalize_embeddings=True, show_progress_bar=False)

        clf = LogisticRegression(max_iter=2000, random_state=42)
        clf.fit(X_train, train_df["label"])

        valid_metrics = self._evaluate(valid_df["label"], clf.predict(X_valid))
        test_metrics = self._evaluate(test_df["label"], clf.predict(X_test))
        overlap = self.compute_overlap(train_df, valid_df, test_df)

        return self._assemble_result_row(
            "MiniLM", train_dataset_label, split_type_label, "statement only",
            train_df, valid_df, test_df, valid_metrics, test_metrics, overlap,
            notes=f"Frozen embeddings, no reweighting/calibration (Uniform-LR equivalent). "
                  f"model={model_name}, batch_size={batch_size}.",
        )

    # ================================================================
    # Convenience: run everything, return xlsx-shaped DataFrame
    # ================================================================

    def run_all_experiments(
        self, liar_df: pd.DataFrame, liar2_df: pd.DataFrame,
        include_transformer_baselines: bool = False,
        include_raw_merge_demo: bool = True,
    ) -> pd.DataFrame:
        """Runs the full testable battery:
            B01/B02  TF-IDF+LogReg on LIAR / LIAR2
            B03/B04  TF-IDF+LogReg on raw merge / clean merge
            B05/B06  TF-IDF+SVM on LIAR / LIAR2
            B07      TF-IDF+SVM on clean merge
            B08/B09  TF-IDF+LogReg+metadata on LIAR / LIAR2
            B10      TF-IDF+LogReg+metadata on clean merge
            C01/C02  TF-IDF+LogReg / SVM on LIAR2 chronological
            B17/B18  Cross-dataset generalization (LIAR->LIAR2, LIAR2->LIAR)
        plus B11-B16 (DistilBERT/MiniLM x 3 datasets) if
        include_transformer_baselines=True.

        B03 (raw merge) is included only as the leakage DEMONSTRATION the
        xlsx itself describes it as -- its notes field says so explicitly;
        don't report it as a result.
        """
        rows = []

        # B01/B02
        print("running B01: TF-IDF + Logistic Regression, train: LIAR; test: LIAR")
        rows.append(self.run_tfidf_logreg(liar_df, "LIAR"))
        print("running B02: TF-IDF + Logistic Regression, train: LIAR2; test: LIAR2")
        rows.append(self.run_tfidf_logreg(liar2_df, "LIAR2"))

        # B03/B04 (merge)
        if include_raw_merge_demo:
            print("running B03: TF-IDF + Logistic Regression, train: LIAR+LIAR2 raw merge; test: LIAR+LIAR2 raw merge")
            raw_merge_df = self.build_liar_liar2_merge(liar_df, liar2_df, deduplicate=False)
            r03 = self.run_tfidf_logreg(raw_merge_df, "LIAR + LIAR2 raw merge")
            r03["notes"] = "Leakage detected -- do not report as final; leakage demonstration only."
            rows.append(r03)

        print("running B04: TF-IDF + Logistic Regression, train: LIAR+LIAR2 clean merge; test: LIAR+LIAR2 clean merge")
        clean_merge_df = self.build_liar_liar2_merge(liar_df, liar2_df, deduplicate=True)
        rows.append(self.run_tfidf_logreg(clean_merge_df, "LIAR + LIAR2 clean merge"))

        # B05/B06/B07
        print("running B05: TF-IDF + Linear SVM, train: LIAR; test: LIAR")
        rows.append(self.run_tfidf_linear_svm(liar_df, "LIAR"))
        print("running B06: TF-IDF + Linear SVM, train: LIAR2; test: LIAR2")
        rows.append(self.run_tfidf_linear_svm(liar2_df, "LIAR2"))
        print("running B07: TF-IDF + Linear SVM, train: LIAR+LIAR2 clean merge; test: LIAR+LIAR2 clean merge")
        rows.append(self.run_tfidf_linear_svm(clean_merge_df, "LIAR + LIAR2 clean merge"))

        # B08/B09/B10
        print("running B08: TF-IDF + Logistic Regression + metadata, train: LIAR; test: LIAR")
        rows.append(self.run_tfidf_logreg_metadata(
            liar_df, "LIAR", self.liar_p.SAFE_METADATA_COLUMNS if self.liar_p else
            ["subject", "speaker", "job_title", "state_info", "party_affiliation", "context"],
        ))
        print("running B09: TF-IDF + Logistic Regression + metadata, train: LIAR2; test: LIAR2")
        rows.append(self.run_tfidf_logreg_metadata(
            liar2_df, "LIAR2", self.liar2_p.SAFE_METADATA_COLUMNS if self.liar2_p else
            ["subject", "speaker", "speaker_description", "state_info", "context"],
        ))
        print("running B10: TF-IDF + Logistic Regression + metadata, train: LIAR+LIAR2 clean merge; test: LIAR+LIAR2 clean merge")
        rows.append(self.run_tfidf_logreg_metadata(
            clean_merge_df, "LIAR + LIAR2 clean merge", self.MERGE_COMMON_METADATA_COLUMNS,
        ))

        # C01/C02 (LIAR2 only)
        print("running C01: TF-IDF + Logistic Regression, train: LIAR2 chronological; test: LIAR2 chronological")
        rows.append(self.run_tfidf_logreg(liar2_df, "LIAR2 chronological", split_mode="chronological"))
        print("running C02: TF-IDF + Linear SVM, train: LIAR2 chronological; test: LIAR2 chronological")
        rows.append(self.run_tfidf_linear_svm(liar2_df, "LIAR2 chronological", split_mode="chronological"))

        # B17/B18 (cross-dataset generalization)
        print("running B17: TF-IDF + Logistic Regression, train: LIAR; test: LIAR2")
        rows.append(self.run_cross_dataset_generalization(liar_df, "LIAR", liar2_df, "LIAR2"))
        print("running B18: TF-IDF + Logistic Regression, train: LIAR2; test - LIAR")
        rows.append(self.run_cross_dataset_generalization(liar2_df, "LIAR2", liar_df, "LIAR"))

        if include_transformer_baselines:
            print("running B11: DistilBERT, train: LIAR; test: LIAR")
            rows.append(self.run_distilbert(liar_df, "LIAR"))
            print("running B12: DistilBERT, train: LIAR2; test: LIAR2")
            rows.append(self.run_distilbert(liar2_df, "LIAR2"))
            print("running B13: DistilBERT, train: LIAR+LIAR2 clean merge; test: LIAR+LIAR2 clean merge")
            rows.append(self.run_distilbert(clean_merge_df, "LIAR + LIAR2 clean merge"))
            print("running B14: MiniLM, train: LIAR; test: LIAR")
            rows.append(self.run_minilm(liar_df, "LIAR"))
            print("running B15: MiniLM, train: LIAR2; test: LIAR2")
            rows.append(self.run_minilm(liar2_df, "LIAR2"))
            print("running B16: MiniLM, train: LIAR+LIAR2 clean merge; test: LIAR+LIAR2 clean merge")
            rows.append(self.run_minilm(clean_merge_df, "LIAR + LIAR2 clean merge"))

        print("\nAll requested experiments complete.")
        return pd.DataFrame(rows)


if __name__ == "__main__":
    print("baseline_models.py loaded. Class: BaselineModels "
          "(methods: build_liar_liar2_merge, run_tfidf_logreg, "
          "run_tfidf_linear_svm, run_tfidf_logreg_metadata, "
          "run_distilbert, run_minilm, run_all_experiments)")

baseline_models.py loaded. Class: BaselineModels (methods: build_liar_liar2_merge, run_tfidf_logreg, run_tfidf_linear_svm, run_tfidf_logreg_metadata, run_distilbert, run_minilm, run_all_experiments)


In [13]:
"""
multifc_preprocessor.py

One class, MultiFC only -- structured the same way as
Liar2Preprocessor/LiarPreprocessor (every step its own method, official
split preserved, leakage-safe metadata). MultiFC is this project's
external-replication dataset: per project methodology, it is evaluated as
INDEPENDENT RETRAINING on MultiFC's own official train/dev/test split,
NOT as zero-shot transfer from a LIAR2-trained model. There is no
chronological split for MultiFC in this class -- retraining uses the
official split directly.

MultiFC's raw TSVs are the 37-site fact-checking corpus (Wang, 2017,
Constructing Datasets for Multi-domain Truth Assessment) -- 13 columns
when labelled (train/dev), 12 when not (test). Its ~26+ site-specific
label taxonomy is harmonized here to 3 classes (false/mixed/true),
matching this project's established label-collapse decisions:
    - ~13-19% of rows are typically dropped as unmappable/unlabelled --
      this must be reported, not silently absorbed (see
      export_attrition_report()).
    - `source_site` (parsed from claim_id's alphabetic prefix) is NOT
      bundled into the default restricted-metadata text -- it's a
      plausible site-identity shortcut and is exposed as its own,
      separate text mode instead, so using it is a deliberate choice,
      not an unquestioned default.
    - `title` and `evidence_text` are excluded from all default text
      modes -- both can directly leak the verdict.

Usage
-----
    from multifc_preprocessor import MultiFCPreprocessor

    p = MultiFCPreprocessor()
    df = p.run_pipeline(train_path, dev_path, test_path)
    p.export_attrition_report("./multifc_reports")
"""

import re
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd


LABELLED_COLUMNS = [
    "claim_id", "claim", "raw_label", "url", "evidence_text",
    "category", "speaker", "checker", "tags", "title",
    "published_date", "claim_date", "entities",
]
UNLABELLED_COLUMNS = [
    "claim_id", "claim", "url", "evidence_text",
    "category", "speaker", "checker", "tags", "title",
    "published_date", "claim_date", "entities",
]


class MultiFCPreprocessor:

    LABEL_ORDER = ["false", "mixed", "true"]
    # Included in model text (restricted metadata, not leakage-risk):
    SAFE_METADATA_COLUMNS = ["category", "speaker", "entities"]
    # NOT included by default -- plausible site-identity shortcut, opt-in only:
    SOURCE_SITE_COLUMN = "source_site"
    # NEVER included -- can directly leak the verdict:
    EXCLUDED_COLUMNS = ["title", "evidence_text", "url", "checker", "tags"]

    def __init__(self):
        self.label_to_id = {label: i for i, label in enumerate(self.LABEL_ORDER)}
        self._build_label_taxonomy()
        self.attrition_report_: dict = {}
        self.unmapped_labels_: dict = {}
        self.class_distribution_: dict = {}

    # ================================================================
    # Label taxonomy (26-site -> 3-class harmonization)
    # ================================================================

    def _build_label_taxonomy(self):
        def norm(x):
            return re.sub(r"\s+", " ", str(x).strip().lower().replace("_", "-"))

        self._true_labels = set(norm(x) for x in [
            "true", "truth!", "verdict: true", "determination: true", "factscan score: true",
            "correct", "accurate", "verified", "fact", "conclusion: accurate",
            "mostly true", "mostly-true", "mostly_true", "mostly truth!", "mostly-correct",
            "determination: mostly true", "promise kept", "in-the-green", "true messages",
            "authorship confirmed!", "confirmed authorship!", "correct attribution",
            "correct attribution!", "truth! & outdated!", "truth! & unproven!",
            "previously truth! now resolved!",
        ])
        self._mixed_labels = set(norm(x) for x in [
            "half-true", "half true", "mixture", "in-between", "partly true", "partially true",
            "needs context", "not the whole story", "compromise", "determination: barely true",
            "truth! & fiction!", "truth! & misleading!", "truth! & disputed!", "disputed!",
            "unproven", "unproven!", "unverified", "determination: misleading",
            "factscan score: misleading", "misleading", "misleading!", "misleading recommendations",
            "2 pinnochios", "a little baloney", "some baloney", "spins the facts",
            "distorts the facts", "cherry picks", "exaggerated", "exaggerates", "understated",
            "half flip", "no flip", "conclusion: unclear", "determination: a stretch",
        ])
        self._false_labels = set(norm(x) for x in [
            "false", "mostly false", "mostly-false", "mostly_false", "pants on fire!",
            "pants-fire", "fiction!", "fiction", "mostly fiction!", "verdict: false",
            "determination: false", "factscan score: false", "conclusion: false",
            "rating: false", "we rate this claim false", "incorrect", "incorrect attribution!",
            "inaccurate attribution!", "fake", "fake news",
        ])
        # Explicitly-known non-mappable categories (distinct from "unrecognized").
        self._drop_labels = set(norm(x) for x in [
            "legend", "satire", "fiction! & satire!", "statirical reports",
            "determination: huckster propaganda",
        ])

    def harmonize_label(self, raw_label) -> Optional[str]:
        """Returns 'true'/'mixed'/'false', or None if unmappable (either an
        explicit drop category, a bare numeric code, or genuinely
        unrecognized). Consult self.unmapped_labels_ after clean_and_filter()
        to distinguish these for reporting."""
        if pd.isna(raw_label):
            return None
        norm = re.sub(r"\s+", " ", str(raw_label).strip().lower().replace("_", "-"))
        if norm in self._true_labels:
            return "true"
        if norm in self._mixed_labels:
            return "mixed"
        if norm in self._false_labels:
            return "false"
        if norm in self._drop_labels:
            return None
        if re.fullmatch(r"\d+", norm):
            return None
        return None

    # ================================================================
    # Step 1: Load, tagging each row with its official split
    # ================================================================

    @staticmethod
    def clean_text(x) -> str:
        """MultiFC's raw TSVs use the literal string "None" (not NaN) for
        empty fields -- treated as empty here, matching what real data
        inspection confirmed."""
        if pd.isna(x):
            return ""
        x = re.sub(r"\s+", " ", str(x)).strip()
        if x.lower() == "none":
            return ""
        return x

    @staticmethod
    def parse_entities(x) -> str:
        """MultiFC's raw entities column is a stringified Python list, e.g.
        "['None']" (no entities) or "['United_States', 'Barack_Obama']".
        Extracts real entity names, dropping the 'None' placeholder."""
        if pd.isna(x):
            return ""
        items = re.findall(r"'([^']*)'", str(x))
        items = [i for i in items if i and i.lower() != "none"]
        return ", ".join(items)

    def _read_tsv(self, path, split_name: str) -> pd.DataFrame:
        raw = pd.read_csv(path, sep="\t", header=None, dtype=str,
                           keep_default_na=False, quoting=3, engine="python")
        if raw.shape[1] == 13:
            raw.columns = LABELLED_COLUMNS
        elif raw.shape[1] == 12:
            raw.columns = UNLABELLED_COLUMNS
            raw["raw_label"] = np.nan
        else:
            raise ValueError(f"{path} has {raw.shape[1]} columns, expected 12 or 13.")
        raw["official_split"] = split_name
        raw["source_file"] = Path(path).name
        raw[self.SOURCE_SITE_COLUMN] = (
            raw["claim_id"].astype(str).str.extract(r"^([A-Za-z]+)", expand=False).fillna("unknown")
        )
        return raw

    def load_raw(self, train_path, dev_path, test_path) -> pd.DataFrame:
        """Load MultiFC's three official TSVs and concatenate, tagging
        each row with `official_split`. Nothing else is cleaned yet."""
        frames = [
            self._read_tsv(train_path, "train"),
            self._read_tsv(dev_path, "dev"),
            self._read_tsv(test_path, "test"),
        ]
        df = pd.concat(frames, ignore_index=True)
        print("Loaded raw MultiFC rows by official_split:")
        print(df["official_split"].value_counts().reindex(["train", "dev", "test"]))
        return df

    # ================================================================
    # Step 2: Text cleaning
    # ================================================================

    def clean_claim_text(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["claim"] = df["claim"].apply(self.clean_text)
        return df

    # ================================================================
    # Step 3: Label harmonization
    # ================================================================

    def normalize_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """Maps raw_label -> harmonized 3-class label. Tracks unmapped
        raw labels per split for later reporting (export_attrition_report)."""
        df = df.copy()
        df["label"] = df["raw_label"].apply(self.harmonize_label)

        for split_name in df["official_split"].unique():
            mask = df["official_split"] == split_name
            unmapped_mask = mask & df["raw_label"].notna() & df["label"].isna()
            counts = df.loc[unmapped_mask, "raw_label"].value_counts()
            if len(counts):
                self.unmapped_labels_[split_name] = counts

        return df

    # ================================================================
    # Step 4: Drop empty claims / unmapped labels, assign label_id
    # ================================================================

    def filter_invalid_rows(self, df: pd.DataFrame, drop_unlabelled: bool = False) -> pd.DataFrame:
        """Drops rows with an empty claim. `drop_unlabelled` controls
        whether rows with no mappable label are also dropped here --
        default False, since the officially unlabelled test split should
        usually be KEPT (with label=NaN) for held-out submission-style
        evaluation, and dropped only for splits you intend to train/
        validate on. Call with drop_unlabelled=True on train/dev
        specifically if you want a train/dev/test-ready DataFrame.
        Records per-split attrition either way."""
        n_before = len(df)
        df = df[df["claim"].str.len() > 0].copy()
        n_after_empty = len(df)

        if drop_unlabelled:
            df = df[df["label"].isin(self.LABEL_ORDER)].copy()
        n_after_label = len(df)

        df["label_id"] = df["label"].map(self.label_to_id)

        for split_name in df["official_split"].unique():
            split_df = df[df["official_split"] == split_name]
            n_split = len(split_df)
            n_labelled = int(split_df["label"].isin(self.LABEL_ORDER).sum())
            dropped_frac = 1 - (n_labelled / n_split) if n_split else 0.0
            self.attrition_report_[split_name] = {
                "n_rows": int(n_split), "n_with_mappable_label": n_labelled,
                "dropped_unmappable_frac": round(dropped_frac, 4),
            }
            print(f"MultiFC {split_name}: {n_labelled}/{n_split} rows have a mappable label "
                  f"({dropped_frac:.1%} unmappable/unlabelled)")

        print(f"filter_invalid_rows: {n_before} -> {n_after_empty} empty-claim rows removed"
              + (f" -> {n_after_label} after dropping unlabelled" if drop_unlabelled else ""))
        return df.reset_index(drop=True)

    # ================================================================
    # Step 5: Build text modes
    # ================================================================

    def _build_claim_only(self, row) -> str:
        return self.clean_text(row.get("claim", ""))

    def _build_claim_plus_safe_metadata(self, row) -> str:
        parts = [self.clean_text(row.get("claim", ""))]
        for col in ("category", "speaker"):
            val = self.clean_text(row.get(col, ""))
            if val:
                parts.append(f"{col}: {val}")
        entities_val = self.parse_entities(row.get("entities", ""))
        if entities_val:
            parts.append(f"entities: {entities_val}")
        return " [SEP] ".join(p for p in parts if p)

    def _build_claim_plus_safe_metadata_and_source(self, row) -> str:
        base = self._build_claim_plus_safe_metadata(row)
        source_val = self.clean_text(row.get(self.SOURCE_SITE_COLUMN, ""))
        if source_val:
            return f"{base} [SEP] source: {source_val}"
        return base

    def build_text_modes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adds:
            model_text__claim_only
            model_text__claim_plus_safe_metadata          (category, speaker, entities)
            model_text__claim_plus_safe_metadata_and_source (+ source_site -- opt-in
                                                              sensitivity-analysis mode,
                                                              a plausible site-identity
                                                              shortcut)
        """
        df = df.copy()
        df["model_text__claim_only"] = df.apply(self._build_claim_only, axis=1)
        df["model_text__claim_plus_safe_metadata"] = df.apply(self._build_claim_plus_safe_metadata, axis=1)
        df["model_text__claim_plus_safe_metadata_and_source"] = df.apply(
            self._build_claim_plus_safe_metadata_and_source, axis=1,
        )
        return df

    # ================================================================
    # Step 6: Drop leakage-risk columns entirely
    # ================================================================

    def drop_leakage_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drops title/evidence_text/url/checker/tags entirely (not just
        excludes them from model text)."""
        present = [c for c in self.EXCLUDED_COLUMNS if c in df.columns]
        df = df.drop(columns=present)
        print(f"drop_leakage_columns: removed {present if present else '(none present)'}")
        return df

    # ================================================================
    # Reporting
    # ================================================================

    def export_attrition_report(self, output_dir) -> None:
        """Writes:
            multifc_attrition_by_split.csv
            multifc_unmapped_labels.csv
            multifc_class_distribution.csv
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        pd.DataFrame(self.attrition_report_).T.to_csv(output_dir / "multifc_attrition_by_split.csv")

        unmapped_rows = []
        for split_name, counts in self.unmapped_labels_.items():
            for raw_label, n in counts.items():
                unmapped_rows.append({"split": split_name, "raw_label": raw_label, "count": n})
        pd.DataFrame(unmapped_rows).to_csv(output_dir / "multifc_unmapped_labels.csv", index=False)

        dist_rows = []
        for stage, dist in self.class_distribution_.items():
            for label, n in dist.items():
                dist_rows.append({"stage": stage, "label": label, "count": n})
        pd.DataFrame(dist_rows).to_csv(output_dir / "multifc_class_distribution.csv", index=False)

        print(f"Wrote attrition/unmapped-label/class-distribution reports to {output_dir}")

    # ================================================================
    # Official split retrieval (no chronological split for MultiFC --
    # independent retraining uses the official split directly)
    # ================================================================

    def get_official_split(self, df: pd.DataFrame):
        """Returns (train_df, dev_df, test_df) using official_split.
        train_df/dev_df are filtered to rows with a mappable label
        (label_id not null); test_df is returned as-is (MultiFC's
        official test partition is unlabelled for the shared-task
        setting -- keep unlabelled rows if you need them, or pre-filter
        yourself if you have gold labels for test)."""
        train_df = df[(df["official_split"] == "train") & df["label"].isin(self.LABEL_ORDER)].copy()
        dev_df = df[(df["official_split"] == "dev") & df["label"].isin(self.LABEL_ORDER)].copy()
        test_df = df[df["official_split"] == "test"].copy()
        return train_df, dev_df, test_df

    # ================================================================
    # Full pipeline
    # ================================================================

    def run_pipeline(self, train_path, dev_path, test_path, drop_unlabelled: bool = False) -> pd.DataFrame:
        """load_raw -> clean_claim_text -> normalize_labels ->
        filter_invalid_rows -> build_text_modes -> drop_leakage_columns"""
        df = self.load_raw(train_path, dev_path, test_path)
        df = self.clean_claim_text(df)
        df = self.normalize_labels(df)
        df = self.filter_invalid_rows(df, drop_unlabelled=drop_unlabelled)
        df = self.build_text_modes(df)
        df = self.drop_leakage_columns(df)

        for split_name in ("train", "dev"):
            split_df = df[df["official_split"] == split_name]
            self.class_distribution_[f"{split_name}_after_filter"] = (
                split_df["label"].value_counts().to_dict()
            )

        print(f"\nrun_pipeline complete: {len(df)} final rows.")
        return df


if __name__ == "__main__":
    print("multifc_preprocessor.py loaded. Class: MultiFCPreprocessor")

multifc_preprocessor.py loaded. Class: MultiFCPreprocessor


In [14]:
"""
multifc_experiment.py

Wires MultiFCPreprocessor (data), DAMiniLMModel (model, class 2 of the
LIAR2 three-class set), and DAMiniLMExperimentRunner (orchestration,
class 3) together for MultiFC specifically -- this is the other half of
this project's central claim: whether DA-MiniLM-RWAC's advantage on
LIAR2 REPRODUCES when independently retrained and evaluated on MultiFC.

Per this project's established framing, MultiFC is evaluated as
INDEPENDENT RETRAINING on its own official train/dev/test split -- NOT as
zero-shot transfer from a LIAR2-trained model. Concretely: DA-MiniLM-RWAC
and its comparators are retrained from scratch on MultiFC's train split,
model-selected on MultiFC's dev split, and evaluated on MultiFC's test
split. There is no chronological condition here (MultiFC has no role in
this project's temporal-drift analysis) -- only the official split.

No modeling logic is reimplemented here. DAMiniLMModel and
DAMiniLMExperimentRunner are reused directly, just configured with
num_classes=3 and MultiFC's label order -- confirmed to work correctly
after fixing a real bug (evaluate_predictions() was silently defaulting
to num_classes=6 for Brier score/NLL, which breaks on any non-6-class
label space via a shape mismatch; now explicit).

Usage
-----
    from multifc_preprocessor import MultiFCPreprocessor
    from da_minilm_model import DAMiniLMModel
    from da_minilm_experiment_runner import DAMiniLMExperimentRunner
    from multifc_experiment import MultiFCExperiment

    prep = MultiFCPreprocessor()
    df = prep.run_pipeline(train_path, dev_path, test_path)

    model = DAMiniLMModel(output_dir="./multifc_results", num_classes=3)
    runner = DAMiniLMExperimentRunner(splitter=None, model=model, num_classes=3)

    exp = MultiFCExperiment(preprocessor=prep, model=model, runner=runner)

    embeddings = {mode: model.encode_texts_minilm(df[f"model_text__{mode}"].tolist(), cache_key=f"multifc_{mode}")
                  for mode in ["claim_only", "claim_plus_safe_metadata"]}

    results_df = exp.run_battery(df, embeddings, seeds=[42, 43, 44, 45, 46],
                                  text_modes=["claim_only", "claim_plus_safe_metadata"])

    ablation_df = exp.run_component_ablation(df, embeddings, text_mode="claim_plus_safe_metadata", seed=42)
"""

from typing import Sequence

import numpy as np
import pandas as pd


class MultiFCExperiment:

    def __init__(self, preprocessor, model, runner):
        """`preprocessor` a MultiFCPreprocessor instance (used for
        get_official_split() -- the only "splitting" MultiFC needs).
        `model` a DAMiniLMModel instance configured with num_classes=3.
        `runner` a DAMiniLMExperimentRunner instance, also configured with
        num_classes=3 (splitter=None is fine -- MultiFC never calls it)."""
        self.p = preprocessor
        self.model = model
        self.runner = runner

    def _assemble_arrays(self, df: pd.DataFrame, embeddings: dict, text_mode: str):
        """Builds the single concatenated (X, y, time_rank, train_idx,
        valid_idx, test_idx) arrays run_single_experiment() expects, from
        MultiFCPreprocessor.get_official_split(). time_rank is all zeros
        -- MultiFC has no temporal condition, but the array is required by
        run_single_experiment()'s signature (inherited from the LIAR2
        pipeline) for interface compatibility."""
        train_df, dev_df, test_df = self.p.get_official_split(df)
        # test_df may include officially-unlabelled rows; keep only labelled
        # ones for evaluation (this project has no gold labels for the
        # official unlabelled MultiFC test partition).
        test_df = test_df[test_df["label_id"].notna()].copy()

        X_train = embeddings[text_mode][train_df.index.values]
        X_valid = embeddings[text_mode][dev_df.index.values]
        X_test = embeddings[text_mode][test_df.index.values]

        y_train = train_df["label_id"].values.astype(int)
        y_valid = dev_df["label_id"].values.astype(int)
        y_test = test_df["label_id"].values.astype(int)

        X_all = np.vstack([X_train, X_valid, X_test])
        y_all = np.concatenate([y_train, y_valid, y_test])
        time_rank_all = np.zeros(len(y_all))

        train_idx = np.arange(len(y_train))
        valid_idx = np.arange(len(y_train), len(y_train) + len(y_valid))
        test_idx = np.arange(len(y_train) + len(y_valid), len(y_all))

        return X_all, y_all, time_rank_all, train_idx, valid_idx, test_idx

    def run_battery(
        self, df: pd.DataFrame, embeddings: dict, seeds: Sequence[int],
        text_modes: Sequence[str], model_kinds: Sequence[str] = ("minilm_logreg", "da_minilm_mlp"),
    ):
        """DA-MiniLM-RWAC / MiniLM-LogReg-RW, retrained from scratch on
        MultiFC's official train, model-selected on dev, evaluated on
        test -- across seeds and text modes. No split_mode loop (MultiFC
        has only the one, official, condition)."""
        all_rows = []
        prediction_store = {}

        for text_mode in text_modes:
            X_all, y_all, time_rank_all, train_idx, valid_idx, test_idx = self._assemble_arrays(
                df, embeddings, text_mode,
            )
            for seed in seeds:
                for model_kind in model_kinds:
                    print(f"running [{model_kind}] MultiFC text={text_mode} seed={seed}")
                    row, lambda_df, pred, proba = self.runner.run_single_experiment(
                        X_all, y_all, time_rank_all, train_idx, valid_idx, test_idx,
                        text_mode=text_mode, split_mode="official_independent_retraining",
                        seed=seed, model_kind=model_kind,
                    )
                    all_rows.append(row)
                    prediction_store[(row["model"], text_mode, seed)] = {
                        "y_test": y_all[test_idx], "pred": pred, "proba": proba,
                    }
                    print(f"  macro_f1={row['macro_f1']:.4f}")

        results_df = pd.DataFrame(all_rows)
        return results_df, prediction_store

    def run_component_ablation(self, df: pd.DataFrame, embeddings: dict, text_mode: str, seed: int,
                                calibration_mode: str = "two_stage") -> pd.DataFrame:
        """The same six-cell ablation (Uniform-LR / LogReg+RW / MiniLM-
        LogReg-RW / Uniform-MLP / MLP+RW / DA-MiniLM-RWAC) run on
        MultiFC's official split -- lets you check whether the reweighting
        and calibration contributions found on LIAR2 hold up on an
        independently-retrained second dataset, not just the model's
        headline comparison."""
        X_all, y_all, time_rank_all, train_idx, valid_idx, test_idx = self._assemble_arrays(
            df, embeddings, text_mode,
        )
        return self.runner.run_component_ablation(
            X_all, y_all, time_rank_all, train_idx, valid_idx, test_idx,
            text_mode=text_mode, split_mode="official_independent_retraining", seed=seed,
            calibration_mode=calibration_mode,
        )


if __name__ == "__main__":
    print("multifc_experiment.py loaded. Class: MultiFCExperiment")

multifc_experiment.py loaded. Class: MultiFCExperiment


# Main experiemtnal code

In [15]:
#from liar2_preprocessor import Liar2Preprocessor

p_liar2 = Liar2Preprocessor()
liar2_dataset = p_liar2.run_pipeline(
    "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/train.csv",
    "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/valid.csv",
    "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/test.csv",
    
)

Loaded raw LIAR2 rows by official_split:
official_split
train    18369
valid     2297
test      2296
Name: count, dtype: int64
filter_invalid_rows: 22962 -> 22962 (dropped 0 empty statements) -> 22962 (dropped 0 unmappable labels)
parse_and_sort_by_date: 22962/22962 rows have a valid date (100.00%) -> sort_source = 'date'
drop_leakage_columns: removed ['justification', 'true_counts', 'mostly_true_counts', 'half_true_counts', 'mostly_false_counts', 'false_counts', 'pants_on_fire_counts']

run_pipeline complete: 22962 final rows.
Official split counts (post-cleaning):
official_split
train    18369
valid     2297
test      2296
Name: count, dtype: int64


In [ ]:
"""
run_all.py

Complete, ordered, end-to-end run of every experiment built in this
project. Paste this into a Kaggle notebook (as one script cell, or split
at the "# ===== SECTION" markers into separate cells) AFTER pasting in
every class file this script imports from:

    liar_preprocessor.py           liar2_preprocessor.py
    baseline_models.py             multifc_preprocessor.py
    temporal_splitter.py           da_minilm_model.py
    da_minilm_experiment_runner.py multifc_experiment.py
    shuffle_control.py             drift_analysis.py

Requires internet enabled on the Kaggle notebook (Settings -> Internet ->
On) for: DA-MiniLM-RWAC/MiniLM-LogReg-RW MiniLM embeddings, DistilBERT
fine-tuning, and MultiFC MiniLM embeddings. A GPU accelerator is strongly
recommended for the MLP/DistilBERT sections.

Update the four path variables in SECTION 0 to match your actual Kaggle
dataset paths before running.
"""

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# SECTION 0: Paths -- EDIT THESE to match your Kaggle input structure
# ============================================================

LIAR_TRAIN = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar dataset/liar dataset/train.tsv"
LIAR_VALID = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar dataset/liar dataset/valid.tsv"
LIAR_TEST = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar dataset/liar dataset/test.tsv"

LIAR2_TRAIN = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/train.csv"
LIAR2_VALID = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/valid.csv"
LIAR2_TEST = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/test.csv"

# Verify these three against your actual MultiFC dataset folder listing
# before running -- filenames/casing vary by upload (e.g. train.tsv vs
# train.tsv.txt). List the folder first if unsure:
#     for p in sorted(Path(MULTIFC_FOLDER).rglob("*")): print(p)
MULTIFC_TRAIN = "/kaggle/input/datasets/badalahmmed/multifc/train.tsv"
MULTIFC_DEV = "/kaggle/input/datasets/badalahmmed/multifc/dev.tsv"
MULTIFC_TEST = "/kaggle/input/datasets/badalahmmed/multifc/test.tsv"

OUTPUT_ROOT = Path("/kaggle/working/full_run_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44, 45, 46]
TEXT_MODES = ["claim_only", "claim_plus_safe_metadata"]

print("=" * 70)
print("SECTION 0 complete: paths configured, output root:", OUTPUT_ROOT)
print("=" * 70)


# ============================================================
# SECTION 1: Load and preprocess LIAR and LIAR2
# ============================================================

#from liar_preprocessor import LiarPreprocessor
#from liar2_preprocessor import Liar2Preprocessor

liar_prep = LiarPreprocessor()
liar_df = liar_prep.run_pipeline(LIAR_TRAIN, LIAR_VALID, LIAR_TEST)

liar2_prep = Liar2Preprocessor()
liar2_df = liar2_prep.run_pipeline(LIAR2_TRAIN, LIAR2_VALID, LIAR2_TEST)

print("=" * 70)
print(f"SECTION 1 complete: LIAR {len(liar_df)} rows, LIAR2 {len(liar2_df)} rows")
print("=" * 70)


# ============================================================
# SECTION 2: Classical + transformer baselines (LIAR, LIAR2, merge,
# chronological, cross-dataset)
# ============================================================

#from baseline_models import BaselineModels

baselines = BaselineModels(liar_preprocessor=liar_prep, liar2_preprocessor=liar2_prep)

# Set to True only if this notebook has internet + GPU enabled.
INCLUDE_TRANSFORMER_BASELINES = True

baseline_results_df = baselines.run_all_experiments(
    liar_df, liar2_df, include_transformer_baselines=INCLUDE_TRANSFORMER_BASELINES,
)
baseline_results_df.to_csv(OUTPUT_ROOT / "baseline_results.csv", index=False)

print("=" * 70)
print(f"SECTION 2 complete: {len(baseline_results_df)} baseline rows saved.")
print("=" * 70)


# ============================================================
# SECTION 3: DistilBERT chronological, multi-seed (a previously
# outstanding required experiment -- only in the baseline battery
# above under the OFFICIAL split, not chronological)
# ============================================================

if INCLUDE_TRANSFORMER_BASELINES:
    distilbert_chrono_df, distilbert_chrono_summary = baselines.run_distilbert_multiseed(
        liar2_df, "LIAR2 chronological", seeds=SEEDS, split_mode="chronological",
    )
    distilbert_chrono_df.to_csv(OUTPUT_ROOT / "distilbert_chronological_multiseed.csv", index=False)
    print("DistilBERT chronological summary:", distilbert_chrono_summary)
else:
    print("Skipping DistilBERT-chronological (INCLUDE_TRANSFORMER_BASELINES=False).")

print("=" * 70)
print("SECTION 3 complete.")
print("=" * 70)


# ============================================================
# SECTION 4: LIAR2 deduplicated + chronologically split cohort
# (used by BOTH the drift diagnostics and the DA-MiniLM experiments,
# matching the notebook this project's central claim is built on)
# ============================================================

audit = liar2_prep.audit_duplicate_statements(liar2_df)
liar2_dedup_df = liar2_prep.drop_duplicate_statements(liar2_df, policy="keep_first", audit_df=audit)
liar2_dedup_df = liar2_dedup_df.reset_index(drop=True)

chrono_train_idx, chrono_valid_idx, chrono_test_idx = liar2_prep.chronological_split(liar2_dedup_df)
drift_train_df = liar2_dedup_df.iloc[chrono_train_idx].copy()
drift_test_df = liar2_dedup_df.iloc[chrono_test_idx].copy()

print(f"Deduplicated chronological cohort: "
      f"train={len(chrono_train_idx)}, valid={len(chrono_valid_idx)}, test={len(chrono_test_idx)}")
print("=" * 70)
print("SECTION 4 complete.")
print("=" * 70)


# ============================================================
# SECTION 5: Drift diagnostics (D01-D05) on the deduplicated
# chronological cohort
# ============================================================

#from drift_analysis import DriftAnalysis

da = DriftAnalysis()

INCLUDE_EMBEDDING_SHIFT = INCLUDE_TRANSFORMER_BASELINES  # D04 needs Hugging Face Hub too

drift_summary_df = da.run_all_drift_experiments(
    drift_train_df, drift_test_df, include_embedding_shift=INCLUDE_EMBEDDING_SHIFT,
)
drift_summary_df.to_csv(OUTPUT_ROOT / "drift_summary.csv", index=False)

drift_comparison_df = da.compare_to_reported(drift_summary_df)
drift_comparison_df.to_csv(OUTPUT_ROOT / "drift_comparison_vs_reported.csv", index=False)

print("=" * 70)
print("SECTION 5 complete: drift diagnostics saved.")
print("=" * 70)


# ============================================================
# SECTION 6: DA-MiniLM-RWAC / MiniLM-LogReg-RW -- the main model
# battery, on the FULL (non-deduplicated) LIAR2 pipeline output,
# matching da-minilm-main-model-v1-0.ipynb exactly (that notebook does
# not deduplicate before its own chronological/random splits).
# ============================================================

#from temporal_splitter import TemporalSplitter
#from da_minilm_model import DAMiniLMModel
#from da_minilm_experiment_runner import DAMiniLMExperimentRunner

liar2_full_df = liar2_df.reset_index(drop=True)
liar2_full_df = liar2_prep.add_time_rank(liar2_full_df)

da_model = DAMiniLMModel(output_dir=str(OUTPUT_ROOT / "da_minilm"), num_classes=6)
da_runner = DAMiniLMExperimentRunner(splitter=liar2_prep, model=da_model, num_classes=6)

# Embeddings, once per text mode (cached to disk automatically by DAMiniLMModel).
da_embeddings = {
    mode: da_model.encode_texts_minilm(liar2_full_df[f"model_text__{mode}"].tolist(), cache_key=mode)
    for mode in TEXT_MODES
}

main_results_df, lambda_selection_df, prediction_store = da_runner.run_full_battery(
    liar2_full_df, da_embeddings, seeds=SEEDS, text_modes=TEXT_MODES,
    split_modes=["chronological", "random_stratified"],
)
main_results_df.to_csv(OUTPUT_ROOT / "da_minilm_main_results.csv", index=False)
lambda_selection_df.to_csv(OUTPUT_ROOT / "da_minilm_lambda_selection.csv", index=False)

summary_df = da_runner.summarize_results(main_results_df, group_cols=["model", "text_mode", "split_mode"])
summary_df.to_csv(OUTPUT_ROOT / "da_minilm_summary.csv", index=False)

print("=" * 70)
print(f"SECTION 6 complete: {len(main_results_df)} DA-MiniLM main-battery rows saved.")
print("=" * 70)


# ============================================================
# SECTION 7: Paired bootstrap -- the statistical test for the central
# claim (DA-MiniLM-RWAC vs MiniLM-LogReg-RW)
# ============================================================

bootstrap_df = da_runner.run_paired_bootstrap(
    prediction_store, text_modes=TEXT_MODES, split_modes=["chronological", "random_stratified"],
    seeds=SEEDS, model_a="DA-MiniLM-RWAC", model_b="MiniLM-LogReg-RW", n_boot=2000,
)
bootstrap_df.to_csv(OUTPUT_ROOT / "da_minilm_paired_bootstrap.csv", index=False)

print("Paired bootstrap results:")
print(bootstrap_df[["text_mode", "split_mode", "seed", "mean_diff", "p_value", "significant_at_95"]])
print("=" * 70)
print("SECTION 7 complete.")
print("=" * 70)


# ============================================================
# SECTION 8: Component ablation -- isolates reweighting's and
# calibration's individual contributions, on the condition carrying
# the central claim (chronological + claim_plus_safe_metadata)
# ============================================================

ABLATION_TEXT_MODE = "claim_plus_safe_metadata"
ABLATION_SPLIT_MODE = "chronological"
ABLATION_SEED = SEEDS[0]

ablation_train_idx, ablation_valid_idx, ablation_test_idx = liar2_prep.chronological_split(liar2_full_df)

ablation_df = da_runner.run_component_ablation(
    da_embeddings[ABLATION_TEXT_MODE],
    liar2_full_df["label_id"].values.astype(int),
    liar2_full_df["time_rank"].values.astype(float),
    ablation_train_idx, ablation_valid_idx, ablation_test_idx,
    text_mode=ABLATION_TEXT_MODE, split_mode=ABLATION_SPLIT_MODE, seed=ABLATION_SEED,
    calibration_mode="two_stage",
)
ablation_df.to_csv(OUTPUT_ROOT / "da_minilm_component_ablation.csv", index=False)

print("Component ablation:")
print(ablation_df[["model", "macro_f1", "use_reweighting", "use_calibration"]])
print("=" * 70)
print("SECTION 8 complete.")
print("=" * 70)


# ============================================================
# SECTION 9: Shuffle-control validation (label permutation null test)
# -- uses the fast LogReg head, 300 shuffles, on the same condition as
# the ablation above
# ============================================================

#from shuffle_control import ShuffleControlValidator

X_shuf = da_embeddings[ABLATION_TEXT_MODE]
y_shuf = liar2_full_df["label_id"].values.astype(int)

X_train_s, y_train_s = X_shuf[ablation_train_idx], y_shuf[ablation_train_idx]
X_valid_s, y_valid_s = X_shuf[ablation_valid_idx], y_shuf[ablation_valid_idx]
X_test_s, y_test_s = X_shuf[ablation_test_idx], y_shuf[ablation_test_idx]


def _shuffle_train_predict_fn(X_tr, y_tr, X_va, y_va, X_te, seed):
    w = np.ones(len(y_tr))
    return da_model.train_logreg_baseline(X_tr, y_tr, w, X_va, y_va, X_te, seed=seed)


shuffle_validator = ShuffleControlValidator(_shuffle_train_predict_fn, n_shuffles=300, seed=42)
shuffle_result = shuffle_validator.run(X_train_s, y_train_s, X_valid_s, y_valid_s, X_test_s, y_test_s)
shuffle_validator.print_summary(shuffle_result)

pd.DataFrame([shuffle_result["summary"]]).to_csv(OUTPUT_ROOT / "shuffle_control_summary.csv", index=False)
np.save(OUTPUT_ROOT / "shuffle_control_null_scores.npy", shuffle_result["null_scores"])

print("=" * 70)
print("SECTION 9 complete.")
print("=" * 70)


# ============================================================
# SECTION 10: Rolling-window temporal evaluation
# ============================================================

rolling_results_df = da_runner.run_rolling_window_evaluation(
    liar2_full_df, da_embeddings, text_mode="claim_plus_safe_metadata",
    seeds=SEEDS, model_kind="da_minilm_mlp",
)
rolling_results_df.to_csv(OUTPUT_ROOT / "da_minilm_rolling_window.csv", index=False)

print("=" * 70)
print(f"SECTION 10 complete: {len(rolling_results_df)} rolling-window rows saved.")
print("=" * 70)


# ============================================================
# SECTION 11: SKIPPED for now (MultiFC's official test split has
# zero labelled rows -- crashes StandardScaler/predict_proba on an
# empty array; needs a dev-set split fix before re-enabling).
# ============================================================
# ============================================================
# SECTION 12: Final summary
# ============================================================

print()
print("=" * 70)
print("FULL RUN COMPLETE. All outputs saved under:", OUTPUT_ROOT)
print("=" * 70)
for f in sorted(OUTPUT_ROOT.rglob("*.csv")):
    print(" ", f.relative_to(OUTPUT_ROOT))

SECTION 0 complete: paths configured, output root: /kaggle/working/full_run_results
Loaded raw LIAR rows by official_split:
official_split
train    10240
valid     1284
test      1267
Name: count, dtype: int64
filter_invalid_rows: 12791 -> 12791 (dropped 0 empty statements) -> 12791 (dropped 0 unmappable labels)
drop_leakage_columns: removed ['barely_true_count', 'false_count', 'half_true_count', 'mostly_true_count', 'pants_on_fire_count']

run_pipeline complete: 12791 final rows.
Official split counts (post-cleaning):
official_split
train    10240
valid     1284
test      1267
Name: count, dtype: int64
Loaded raw LIAR2 rows by official_split:
official_split
train    18369
valid     2297
test      2296
Name: count, dtype: int64
filter_invalid_rows: 22962 -> 22962 (dropped 0 empty statements) -> 22962 (dropped 0 unmappable labels)
parse_and_sort_by_date: 22962/22962 rows have a valid date (100.00%) -> sort_source = 'date'
drop_leakage_columns: removed ['justification', 'true_counts', 'm

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Saved embeddings: /kaggle/working/full_run_results/da_minilm/embeddings_claim_only.npy (22962, 384)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/180 [00:00<?, ?it/s]

Saved embeddings: /kaggle/working/full_run_results/da_minilm/embeddings_claim_plus_safe_metadata.npy (22962, 384)
running [minilm_logreg] text=claim_only split=chronological seed=42
  macro_f1=0.2318, ece=0.0622, lambda=0.05
running [da_minilm_mlp] text=claim_only split=chronological seed=42
  macro_f1=0.2389, ece=0.0852, lambda=0.2
running [minilm_logreg] text=claim_only split=chronological seed=43
  macro_f1=0.2318, ece=0.0622, lambda=0.05
running [da_minilm_mlp] text=claim_only split=chronological seed=43
  macro_f1=0.2342, ece=0.0749, lambda=0.1
running [minilm_logreg] text=claim_only split=chronological seed=44
  macro_f1=0.2318, ece=0.0622, lambda=0.05
running [da_minilm_mlp] text=claim_only split=chronological seed=44
  macro_f1=0.2510, ece=0.1320, lambda=0.3
running [minilm_logreg] text=claim_only split=chronological seed=45
  macro_f1=0.2318, ece=0.0622, lambda=0.05
running [da_minilm_mlp] text=claim_only split=chronological seed=45
  macro_f1=0.2433, ece=0.0571, lambda=0.0
ru

In [ ]:
from pathlib import Path
import zipfile

working_dir = Path("/kaggle/working")
output_zip = working_dir / "kaggle_working_backup.zip"

with zipfile.ZipFile(
    output_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zip_file:

    for path in working_dir.rglob("*"):
        if not path.is_file():
            continue

        # Prevent the ZIP file from adding itself.
        if path.resolve() == output_zip.resolve():
            continue

        archive_name = path.relative_to(working_dir)
        zip_file.write(path, arcname=archive_name)

print(f"ZIP created successfully: {output_zip}")
print(f"Size: {output_zip.stat().st_size / (1024**2):.2f} MB")